# Cosmos3-Nano DCA-Stable **v2** — measurement-fixed sweep

Successor to `Cosmos3_Nano_DCA_stable_sweep_vastai.ipynb`. Same DCA idea, same 4-rung ladder,
but the **measurement layer is rebuilt** after the P1/P2 post-mortem showed the old one could not
tell a learning model from a frozen one.

**What the P1/P2 analysis found (all verified on the 1242-frame test split):**

| Finding | Evidence |
|---|---|
| Direction head is **worse than a constant** | trivial "always straight" = 17.22°, best rung = 18.51° |
| Velocity head barely beats a constant | predict-median = 0.308 MAE, best rung = 0.285 |
| Turn side is **at/below chance** | `turning_right` is 70% *negative* dir_deg — side is not in the heading vector |
| `turn_side_sign_acc` was 68% padding | `RARE` included `walking_fast` (215 of 314 frames), which passes trivially |
| `scene_unsafe_go` rewards paralysis | the **human demonstrator scores 0.675**; P1's "great" 0.043 was a frozen model (worst vel_mae in the sweep) |
| `safe_speed_corr` rewards paralysis | the **human scores 0.434**; P1's 0.708–0.878 is over-coupling |
| 3 of 6 ladder metrics are rung-invariant | affordance losses are active in *all* rungs and read the raw latent |
| `unsafe_go` is redundant | `unsafe_go == 1 - stop_recall` exactly, so fig 16 plotted `y = 1-x` |
| The split was **not** stratified | `RARE` contained `walking_fast` (27% of frames) → 24/25 test clips in one stratum |
| Safety stats had effective n ≈ 41 | 91 stop frames from 6 clips (81% from 3); P1 vs P2 gap = 0.6 SE |
| `L_false` was **dead code** | `relu(0.15 - v_pred)` never fires; rung V1 added only one live term |
| Mode and velocity heads contradict | ordinal head's uniform expectation was **0.600** > `go_speed` 0.50, and it could not emit 0 |

**What v2 changes** — see `DCA_STABLE_V2_CHANGES.md` for the full rationale.

1. **Every metric ships with its trivial-predictor reference**, computed from the ground truth of the
   same split inside `evaluate()`. Figures draw the reference line.
2. **Genuinely stratified 3-way split** (train/val/test) on `(has_stop, has_turn_or_step)`.
3. **6-class mode head** — turn/sidestep side is *classified*, not decoded from `sin`.
4. **5-bin ordinal velocity with a true-zero bin** + class-balanced band weights.
5. **Mode↔velocity consistency loss** — `p_stand · v`.
6. **Demonstrator-relative safety** (`unsafe_relative`) by default, so `L_false` is alive.
7. **Per-epoch validation + best-checkpoint selection.** The old run had no val set at all.
8. **~4× faster**: cached frozen `z₀` + streamed temporal pairs → 1 backbone forward per sample
   (was 4 for `dca_full`).
9. **Figures rebuilt** — 10 concrete bugs fixed, plus a render-time smoke test that fails the run
   if a figure's aspect ratio blows up again.

Run the **CPU SELF-TEST** cell first — it validates every phase × rung path with no GPU.

## Part A — Install + Environment
Installs pinned deps, patches transformers version check without importing it. Uses `subprocess` (no `!pip`).

In [ ]:
import os, subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-warn-script-location"] + list(args))

# Cosmos3-Nano's reasoner is Qwen3-VL arch. The Cosmos3 Transformers integration first
# ships in transformers>=5.11.0 (verified on the Kaggle feasibility check). Let pip
# RESOLVE deps -- do NOT use --no-deps (on Cosmos3 that caused a version-window whack-a-mole).
# Do NOT reinstall torch/torchvision -> start from a recent nvcr.io/pytorch base on Vast.ai.
pip("-U", "transformers>=5.11.0", "peft>=0.13.2", "accelerate>=0.34.2",
    "safetensors>=0.8.0", "av", "pillow", "regex", "tqdm", "pyyaml", "packaging",
    "requests", "scikit-learn", "matplotlib")

# Base images preinstall torchao 0.10.0; transformers 5.11 rejects torchao<0.16 the moment
# it detects it, and we don't use torchao -> remove it so the guard never fires.
subprocess.call([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"])

# Qwen3-VL's processor stack builds Qwen3VLVideoProcessor, which HARD-REQUIRES torchvision
# from transformers 5.11 onward. Several nvcr.io PyTorch bases ship torch WITHOUT torchvision,
# so AutoProcessor.from_pretrained() dies with an ImportError before the model ever loads.
# Install the matching build with --no-deps so torch itself is never touched or downgraded.
def _has_torchvision():
    return subprocess.call([sys.executable, "-c", "import torchvision"],
                           stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL) == 0

if _has_torchvision():
    print("torchvision: present")
else:
    import torch
    base, _, local = torch.__version__.partition("+")
    tv = f"0.{int(base.split('.')[1]) + 15}"      # torch 2.N  <->  torchvision 0.(N+15)
    print(f"torchvision missing; installing {tv}.* to match torch {torch.__version__}")
    attempts = []
    if local.startswith("cu"):
        attempts.append(["--no-deps", f"torchvision=={tv}.*",
                         "--index-url", f"https://download.pytorch.org/whl/{local}"])
    attempts.append(["--no-deps", f"torchvision=={tv}.*"])          # PyPI fallback
    attempts.append(["--no-deps", "torchvision"])                   # last resort
    for args in attempts:
        try: pip(*args)
        except Exception as e: print("  attempt failed:", str(e)[:120]); continue
        if _has_torchvision(): break
    assert _has_torchvision(), (
        "torchvision still not importable. Install a build matching "
        f"torch {torch.__version__} manually, then re-run.")
    v = subprocess.check_output(
        [sys.executable, "-c", "import torchvision;print(torchvision.__version__)"]).decode().strip()
    print("torchvision installed:", v)

print("all deps ready (Cosmos3-Nano / transformers>=5.11.0, torchao removed)")

In [ ]:
import torch
print("torch", torch.__version__, "| CUDA", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no gpu")
assert torch.cuda.is_available(), "Need a GPU"

## Part B — Config
All 4 rungs queued. `PHASES_TO_RUN` defaults to `V2_full` (every science fix on).

In [ ]:
import os

def _rank_for(hidden, anchor_rank=16, anchor_hidden=4096):
    """Locked capacity rule: rank scales with backbone width, rounded to mult of 8,
    anchored on OpenVLA (r=16 @ hidden 4096). 4096 -> 16."""
    r = round(anchor_rank * hidden / anchor_hidden)
    return max(8, int(round(r / 8) * 8))

COSMOS_HIDDEN = 4096                    # confirmed from Cosmos3-Nano config (text_config = qwen3_vl_text)
LORA_RANK = _rank_for(COSMOS_HIDDEN)    # -> 16

CFG = {
    "backbone":       "cosmos3-nano",
    "repo_id":        "s-alam/dense_walk_vid_new_temp",
    "json_dir":       "json_openvla",
    "keyframes_dir":  "keyframes",
    # NOTE: CLAUDE.md prescribes /workshop (accelerate/device-map issue on some images).
    # P1/P2 ran fine on /workspace. Flip these two if the base image misbehaves.
    "local":          "/workspace/dw_data",
    "out":            "/workspace/dw_out_cosmos3_v2",
    "base_model":     "nvidia/Cosmos3-Nano",

    # ---- split (v2: 3-way, genuinely stratified) --------------------------------
    "test_frac":      0.20,
    "val_frac":       0.10,          # set 0.0 to reproduce the old 2-way split
    "split_seed":     42,

    # ---- optimisation -----------------------------------------------------------
    "epochs":         5,
    "lr":             5e-4,          # used only when lr_sched is OFF
    "lr_stable":      1.5e-4,        # used when lr_sched is ON
    "warmup_frac":    0.08,
    "grad_clip":      1.0,
    "grad_accum":     8,
    "lora_rank":      LORA_RANK,
    "lora_alpha":     LORA_RANK * 2,
    "lora_dropout":   0.0,
    "temporal_pairs": True,
    "n_interleave":   8,             # distinct clips mixed inside one grad-accum window

    # ---- loss weights -----------------------------------------------------------
    "lambda": {"vel":1.0, "dir":1.0, "mode":1.0, "unsafe":2.0, "false":0.5,
               "gate":0.5, "prior":0.5, "temp":0.2, "consist":0.5,
               "obs_dist":1.0, "obs_close":1.0, "trav":0.5},

    # ---- velocity banding (v2: 5 bins WITH a true-zero bin) ---------------------
    # v1 was reps=[0.075,0.325,0.75,1.25]: it could not emit 0, and its uniform-softmax
    # expectation was exactly 0.600 -- above go_speed 0.50 -- so an *uncertain* model was
    # scored unsafe by default. That is the mechanism behind P2's scene_unsafe_go = 0.837.
    # Measured on the 100-clip train split: a (0.02, 0.15] crawl bin held 10 of 3212 frames
    # (0.31%) and, being rarest, drew the LARGEST class-balanced weight (4.50x) -- a 77x
    # spread across bins. Merged into the slow band: all four bins are now populated and the
    # weight spread drops to 4.6x. The true-zero bin (which is exactly the 182 stop frames)
    # is what mattered and is kept. Revert to [0.02,0.15,0.50,1.00] / 5 reps to undo.
    "vel_edges":      [0.02, 0.50, 1.00],
    "vel_reps":       [0.0, 0.30, 0.75, 1.25],
    "vel_reps_from_data": True,   # replace anchors with the per-bin mean of the train split

    # ---- safety / affordance ----------------------------------------------------
    "eps_stop":       0.05,
    "close_m":        2.0,
    "obs_norm_m":     10.0,
    "go_speed":       0.50,   # legacy absolute threshold -- the HUMAN violates it 67.5% of
                              # the time on close frames, so it is reported, never optimised
    "crawl_speed":    0.15,
    "rel_margin":     0.15,   # deadband for demonstrator-relative safety
    "temp_alpha":     1.0,
    "eta_cls":        1e-3,
    "cb_beta":        0.999,
    "focal_gamma":    2.0,
    "relabel_max_slow": 0.7,

    # ---- direction reweighting --------------------------------------------------
    # A constant "straight ahead" beats every P1/P2 rung (17.22 deg vs 18.51-20.35).
    # Frames with dir ~ 0 are exactly where the constant wins and there is nothing to
    # learn; concentrate the loss on the informative ones instead of averaging it away.
    "dir_floor_w":    0.2,    # weight kept on straight-ahead frames
    "dir_ref_deg":    30.0,   # |dir| at which the weight saturates to 1.0

    # ---- validation / checkpoint selection --------------------------------------
    "val_every":      1,
    "val_max_frames": 600,    # subsample val for speed; None = all
    "select_metric":  "macro_f1",
    "select_mode":    "max",

    # ---- compute ----------------------------------------------------------------
    "fast_path":      True,   # cache frozen z0 + stream temporal pairs (~4x)
    "rungs": ["lora_only", "lora_unsafe", "lora_unsafe_prior", "dca_full"],
}

ABLATION_FLAGS = {
    "lora_only":         dict(unsafe=False, prior=False, gate=False, temp=False),
    "lora_unsafe":       dict(unsafe=True,  prior=False, gate=False, temp=False),
    "lora_unsafe_prior": dict(unsafe=True,  prior=True,  gate=False, temp=False),
    "dca_full":          dict(unsafe=True,  prior=True,  gate=True,  temp=True),
}

# ---- HF download robustness -------------------------------------------------
# The mini/full datasets are ~10k-20k small files. At the default concurrency the Xet
# token endpoint returns HTTP 429 partway through. Lower concurrency + retry with
# backoff; snapshot_download RESUMES, so each attempt keeps prior progress.
CFG["hf_workers"]     = 4       # default is 8; 429s scale with concurrency
CFG["hf_retries"]     = 8
CFG["hf_disable_xet"] = False   # flip to True if 429s persist (forces plain HTTPS)

# must be set before huggingface_hub is first imported
os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "0")
if CFG["hf_disable_xet"]:
    os.environ["HF_HUB_DISABLE_XET"] = "1"

HF_TOKEN = os.environ.get("HF_TOKEN")
assert HF_TOKEN, "Set HF_TOKEN env var before running:  export HF_TOKEN=hf_xxx"

os.makedirs(CFG["local"], exist_ok=True)
os.makedirs(CFG["out"], exist_ok=True)

# ============================================================================
# FIX FLAGS + CUMULATIVE PHASES
# ============================================================================
# SCIENCE fixes -- change what the model learns / how it is scored.
FIX_GROUPS = {
    "stability":    dict(grad_clip=True, lr_sched=True, huber_pen=True,
                         epoch_flush=True, afford_raw=True),
    "learnability": dict(cb_focal=True, ordinal_vel=True, cb_vel=True),
    # the vel<->unsafe conflict fix: unsafe/false measure against the demonstrator's own
    # speed at that frame instead of two absolute constants. Also the reason L_false stops
    # being dead code -- relu(0.15 - v_pred) never fired, relu((v_t - m) - v) does.
    "conflict":     dict(unsafe_relative=True),
    # v2 decode group: side is classified not decoded; velocity can express zero;
    # mode and velocity are tied together; L_dir focuses on informative frames.
    "decode":       dict(mode6=True, consist=True, dir_reweight=True),
    "science":      dict(safe_relabel=True, uncert_weight=True),
}
# COMPUTE fixes -- numerically near-equivalent, ~4x faster. Applied to every phase when
# CFG["fast_path"] is True so they never confound a science comparison.
SPEED_FIXES = dict(cache_z0=True, stream_pairs=True)

def _cum(*groups):
    d = {}
    for g in groups: d.update(FIX_GROUPS[g])
    if CFG["fast_path"]: d.update(SPEED_FIXES)
    return d

PHASE_CONFIGS = {
    "P0_baseline":     dict(SPEED_FIXES) if CFG["fast_path"] else {},
    "P1_stability":    _cum("stability"),
    "P2_learnability": _cum("stability", "learnability"),
    "P3_conflict":     _cum("stability", "learnability", "conflict"),
    "P4_decode":       _cum("stability", "learnability", "conflict", "decode"),
    "V2_full":         _cum("stability", "learnability", "conflict", "decode"),
    "V2_science":      _cum("stability", "learnability", "conflict", "decode", "science"),
}

# EDIT THESE TWO LINES TO CONTROL COMPUTE:
PHASES_TO_RUN = ["baseline", "decouple_only", "all_fixes"]
RUNGS_TO_RUN  = ["lora_only", "dca_full"]

FIX = {}   # active fix toggles for the current phase (sweep loop mutates this)
DEV = "cuda:0"

print(f"config ready | backbone={CFG['backbone']} | lora_rank={CFG['lora_rank']} (alpha={CFG['lora_alpha']})")
print("phases queued:", PHASES_TO_RUN)
print("rungs  queued:", RUNGS_TO_RUN)
print("fast_path:", CFG["fast_path"], "| vel bins:", CFG["vel_reps"],
      f"(uniform expectation {sum(CFG['vel_reps'])/len(CFG['vel_reps']):.3f} vs go_speed {CFG['go_speed']
    # v3 diagnostic phases
    "baseline":        _cum("stability", "learnability", "conflict", "decode"),
    "decouple_only":   _cum("stability", "learnability", "conflict", "decode"),
    "all_fixes":       _cum("stability", "learnability", "conflict", "decode"),
})")

In [ ]:
TEST_CONFIGS = {
    'baseline': {
        'afford_decouple': False,
        'use_cagrad': False,
        'lora_rank_override': None,
        'lora_dropout': 0.0,
        'lambda_obs_dist': 1.0,
        'lambda_obs_close': 1.0,
        'lambda_trav': 0.5,
    },
    'decouple_only': {
        'afford_decouple': True,
        'use_cagrad': False,
        'lora_rank_override': None,
        'lora_dropout': 0.15,
        'lambda_obs_dist': 0.1,
        'lambda_obs_close': 0.1,
        'lambda_trav': 0.05,
    },
    'all_fixes': {
        'afford_decouple': True,
        'use_cagrad': True,
        'lora_rank_override': 8,
        'lora_dropout': 0.15,
        'lambda_obs_dist': 0.1,
        'lambda_obs_close': 0.1,
        'lambda_trav': 0.05,
    }
}


## Part C — Data + Split
Pulls golden/uni JSON from HF, derives the 6-class mode, and makes a **genuinely stratified**
train/val/test split.

In [ ]:
import json, math, random, glob, contextlib, time
from collections import Counter, defaultdict
import numpy as np
import torch
from huggingface_hub import snapshot_download

# Exact patterns. A bare "keyframes*/**" also matches the repo's SUPERSEDED keyframes_old/
# tree (~4.7k files) and doubles the download to ~9.6k files -- which is what triggered an
# HTTP 429 mid-transfer. Shard support is explicit (_gpu0, _gpu1, ...) instead of a bare *.
ALLOW_PATTERNS = [
    CFG["json_dir"] + "/*",             # json_openvla/*
    CFG["json_dir"] + "_gpu*/*",        # json_openvla_gpu0/*  (sharded full dataset)
    CFG["keyframes_dir"] + "/**",       # keyframes/**
    CFG["keyframes_dir"] + "_gpu*/**",  # keyframes_gpu0/**
]
IGNORE_PATTERNS = ["*_old/**", "*_old/*"]   # belt-and-braces against json_a_old/ etc.

_TRANSIENT = ("429", "too many requests", "connectionerror", "connection reset",
              "connection aborted", "timed out", "timeout", "502", "503", "504",
              "incompleteread", "remote end closed")

def robust_snapshot():
    """snapshot_download with backoff. Resumes -- files already on disk are skipped, so a
    retry costs only the remainder. Raises immediately on non-transient errors (auth, 404)."""
    delay = 15
    for attempt in range(1, CFG["hf_retries"] + 1):
        try:
            snapshot_download(
                repo_id=CFG["repo_id"], repo_type="dataset", local_dir=CFG["local"],
                allow_patterns=ALLOW_PATTERNS, ignore_patterns=IGNORE_PATTERNS,
                token=HF_TOKEN, max_workers=CFG["hf_workers"],
            )
            print(f"  snapshot complete (attempt {attempt})")
            return
        except Exception as e:
            msg = f"{type(e).__name__}: {e}"
            if attempt == CFG["hf_retries"] or not any(t in msg.lower() for t in _TRANSIENT):
                raise
            print(f"  [hf retry {attempt}/{CFG['hf_retries']}] {msg[:180]}")
            print(f"  sleeping {delay}s, then resuming (downloaded files are kept)", flush=True)
            time.sleep(delay)
            delay = min(delay * 2, 300)

robust_snapshot()
_kf_dirs = sorted(d for d in glob.glob(os.path.join(CFG["local"], CFG["keyframes_dir"] + "*"))
                  if os.path.isdir(d))
_n_kf = sum(len(fs) for d in _kf_dirs for _, _, fs in os.walk(d))
print("keyframe dirs:", [os.path.basename(d) for d in _kf_dirs], f"| {_n_kf} images")
assert not any(d.endswith("_old") for d in _kf_dirs),     "a *_old/ tree was downloaded -- ALLOW_PATTERNS over-matched (this doubles the transfer)"

json_files = sorted(glob.glob(os.path.join(CFG["local"], CFG["json_dir"] + "*", "*.json")))
print("golden/uni JSON files:", len(json_files))
assert json_files, "No golden/uni JSONs found under any json_openvla* dir."

# ── canonical labels (v5 grounded pipeline: 8 commands) ──
MODE_IDS    = {"stand": 0, "walk": 1, "turn": 2, "sidestep": 3}
STOP_LABELS = {"standing_still"}

# v2: 6-class mode. Turn/sidestep SIDE is part of the class, not decoded from sin(dir).
# Verified on the P1/P2 test split: turning_right frames are 70% NEGATIVE dir_deg, so the
# old `sin < 0 -> left` rule was guaranteed to misclassify most right turns. Sidesteps do
# separate cleanly (step_left 100% negative, step_right 0%), but turns do not -- turn side
# lives in yaw_rate, not in the heading vector. Classifying it removes the decode entirely.
MODE6_IDS = {"stand":0, "walk":1, "turn_left":2, "turn_right":3, "step_left":4, "step_right":5}
MODE6_NAMES = [k for k, _ in sorted(MODE6_IDS.items(), key=lambda kv: kv[1])]
MODE6_FROM_LABEL = {
    "standing_still": 0,
    "walking_slow": 1, "walking_normal": 1, "walking_fast": 1,
    "turning_left": 2, "turning_right": 3,
    "stepping_left": 4, "stepping_right": 5,
}

# genuinely rare classes. v1 put walking_fast in here; it is 27% of all frames, which made
# the "stratified" split a no-op (24/25 test clips landed in the same stratum).
TURN_STEP = {"turning_left", "turning_right", "stepping_left", "stepping_right"}
RARE      = set(TURN_STEP)

BANDS = {"stop_v": 0.15, "band_slow": 0.50, "band_normal": 1.00, "yaw_turn": 7.0}

def resolve_img(p):
    base = os.path.basename(p)
    vid  = os.path.basename(os.path.dirname(p))
    cands = [p, os.path.join(CFG["local"], p)]
    for d in glob.glob(os.path.join(CFG["local"], CFG["keyframes_dir"] + "*")):
        cands.append(os.path.join(d, vid, base))
    for c in cands:
        if os.path.exists(c): return c
    return cands[1]

def load_clip(path):
    d = json.load(open(path))
    instr = d["instruction"]["text"] if isinstance(d["instruction"], dict) else d["instruction"]
    frames, n_dropped = [], 0
    for f in d["frames"]:
        # solve_failed = degenerate ego-motion geometry; the stored motion is a fabricated
        # all-zero placeholder, not a measurement -> drop rather than train on a fake target.
        if "solve_failed" in (f.get("flags") or []):
            n_dropped += 1
            continue
        a = f["action"]; lab = a["label"]
        img = resolve_img(f["image"])
        if not os.path.exists(img): continue
        nav  = f.get("navigability") or {}
        _nod = nav.get("nearest_obstacle_m")
        obs_valid = isinstance(_nod, (int, float)) and _nod >= 0
        obs_dist  = float(_nod) if obs_valid else 0.0
        obs_close = 1.0 if (obs_valid and obs_dist < CFG["close_m"]) else 0.0
        _sb  = nav.get("surface_blocked_pct")
        trav_valid = isinstance(_sb, (int, float))
        frames.append({
            "image": img, "label": lab,
            "v": float(a["velocity_mps"]), "dir_deg": float(a["direction_deg"]),
            "mode_id":  MODE_IDS[a["mode"]],
            "mode6_id": MODE6_FROM_LABEL[lab],
            "rho":      obs_close,
            "stop":     1.0 if a.get("is_stop", lab in STOP_LABELS) else 0.0,
            "obs_dist": obs_dist, "obs_close": obs_close, "obs_valid": float(obs_valid),
            "trav": float(_sb) if trav_valid else 0.0, "trav_valid": float(trav_valid),
            "weight": float(f.get("weight", 1.0)),
        })
    return {"video_id": d.get("video", d.get("video_id")),
            "instruction": instr, "frames": frames, "n_dropped": n_dropped}

_loaded = [load_clip(p) for p in json_files]
clips = [c for c in _loaded if len(c["frames"]) > 0]
print("clips:", len(clips), "| frames:", sum(len(c["frames"]) for c in clips),
      "| dropped (solve_failed):", sum(c["n_dropped"] for c in _loaded))

# ── STRATIFIED 3-WAY SPLIT ────────────────────────────────────────────────────
# v1 stratum was `"rare=" + any(label in RARE)` with walking_fast inside RARE -> constant
# for 24/25 clips, i.e. an unstratified split. Consequence: the 91 test stop frames landed
# in 6 of 25 clips (81% in 3), giving every safety metric an effective n of ~41.
# v2 stratifies on the two things that actually drive the safety and turn metrics.
def stratum(c):
    labs = {f["label"] for f in c["frames"]}
    return (bool(labs & STOP_LABELS), bool(labs & TURN_STEP))

by = defaultdict(list)
for c in clips: by[stratum(c)].append(c)

rng = random.Random(CFG["split_seed"])
train_clips, val_clips, test_clips = [], [], []
for k, g in sorted(by.items()):
    g = g[:]; rng.shuffle(g)
    n     = len(g)
    n_te  = max(1, round(n * CFG["test_frac"])) if n > 2 else (1 if n > 1 else 0)
    n_va  = max(1, round(n * CFG["val_frac"]))  if (CFG["val_frac"] > 0 and n - n_te > 1) else 0
    test_clips  += g[:n_te]
    val_clips   += g[n_te:n_te + n_va]
    train_clips += g[n_te + n_va:]

def _summary(name, cs):
    fs = [f for c in cs for f in c["frames"]]
    n_stop = sum(1 for f in fs if f["label"] in STOP_LABELS)
    n_ts   = sum(1 for f in fs if f["label"] in TURN_STEP)
    n_sc   = sum(1 for c in cs if any(f["label"] in STOP_LABELS for f in c["frames"]))
    print(f"  {name:6s} clips {len(cs):4d} | frames {len(fs):6d} | stop {n_stop:5d} "
          f"(in {n_sc} clips) | turn/step {n_ts:5d}")

print(f"strata: {dict((str(k), len(v)) for k, v in sorted(by.items()))}")
for nm, cs in [("train", train_clips), ("val", val_clips), ("test", test_clips)]:
    _summary(nm, cs)
assert train_clips and test_clips, "empty train or test split"

# ── POWER WARNING ──────────────────────────────────────────────────────────────
# Every stop-based metric is computed over these frames only; correlated frames inside a
# clip mean the effective n is roughly half the nominal count. P1/P2 had 91 stop frames in
# 6 clips -> effective n ~41, which is why their stop_recall gap of 0.033 was 0.6 SE.
_ts   = [f for c in test_clips for f in c["frames"] if f["label"] in STOP_LABELS]
_tsc  = len({id(c) for c in test_clips if any(f["label"] in STOP_LABELS for f in c["frames"])})
print(f"\nPOWER: {len(_ts)} test stop frames across {_tsc} clips.")
if len(_ts) < 150 or _tsc < 10:
    print("  !! WARNING: stop-based metrics (stop_recall / unsafe_go) will have wide CIs.")
    print("     Independent CLIPS are what buy power here, not frames per clip.")

# ── class weights (TRAIN only) ────────────────────────────────────────────────
def _cb_weights(freq, beta):
    w = (1.0 - beta) / (1.0 - np.power(beta, np.maximum(freq, 1)))
    return w / w.mean()

_tr_frames = [f for c in train_clips for f in c["frames"]]
mc   = Counter(f["mode_id"] for f in _tr_frames)
freq = np.array([mc.get(i, 0) for i in range(4)], dtype=np.float64)
_iw  = 1.0 / (freq + CFG["eta_cls"]); _iw = _iw / _iw.mean()
MODE_W    = torch.tensor(_iw, dtype=torch.float32)
CB_MODE_W = torch.tensor(_cb_weights(freq, CFG["cb_beta"]), dtype=torch.float32)

mc6   = Counter(f["mode6_id"] for f in _tr_frames)
freq6 = np.array([mc6.get(i, 0) for i in range(6)], dtype=np.float64)
_iw6  = 1.0 / (freq6 + CFG["eta_cls"]); _iw6 = _iw6 / _iw6.mean()
MODE6_W    = torch.tensor(_iw6, dtype=torch.float32)
CB_MODE6_W = torch.tensor(_cb_weights(freq6, CFG["cb_beta"]), dtype=torch.float32)

# v2: the velocity-band CE was UNWEIGHTED in v1, so the ordinal head drifted to the
# majority band. The zero bin is the rarest and the most safety-relevant -> weight it.
_vt    = np.array([f["v"] for f in _tr_frames])
_vb    = np.digitize(_vt, CFG["vel_edges"])
freqv  = np.array([int((_vb == i).sum()) for i in range(len(CFG["vel_reps"]))], dtype=np.float64)

# anchors from the data, not hand-picked: each bin's representative becomes the mean speed
# actually observed in it (bins with <20 frames keep the configured anchor).
if CFG.get("vel_reps_from_data"):
    _reps = [float(_vt[_vb == i].mean()) if freqv[i] >= 20 else CFG["vel_reps"][i]
             for i in range(len(CFG["vel_reps"]))]
    print("vel reps: configured", [round(x,3) for x in CFG["vel_reps"]],
          "-> empirical", [round(x,3) for x in _reps])
    CFG["vel_reps"] = _reps

CB_VEL_W = torch.tensor(_cb_weights(freqv, CFG["cb_beta"]), dtype=torch.float32)

# a class holding <1% of frames but carrying the top weight destabilises training --
# that is exactly what the 5-bin velocity scheme did before the bins were merged.
def _warn_sparse(name, labels, freq, weights):
    tot = max(1.0, float(sum(freq)))
    for lbl, f, w in zip(labels, freq, weights):
        if f / tot < 0.01:
            print(f"  !! SPARSE {name}: '{lbl}' has {int(f)} frames ({100*f/tot:.2f}%) "
                  f"but weight {w:.2f} -- consider merging or capping")
_warn_sparse("vel band", [f"bin{i}" for i in range(len(freqv))], freqv, CB_VEL_W.tolist())
_warn_sparse("mode6", MODE6_NAMES, freq6, CB_MODE6_W.tolist())

print("\nmode4 freq:", dict(mc), "| CB w:", [round(x,3) for x in CB_MODE_W.tolist()])
print("mode6 freq:", {MODE6_NAMES[i]: int(freq6[i]) for i in range(6)})
print("mode6 CB w:", [round(x,3) for x in CB_MODE6_W.tolist()])
print("vel band freq:", freqv.astype(int).tolist(), "| CB w:", [round(x,3) for x in CB_VEL_W.tolist()])

LABEL_MAP = json.load(open(json_files[0]))["action_space"]["label_map"]
LABELS    = list(LABEL_MAP.keys())
print("labels:", LABELS)

## Part D — Model Factory + DCA Heads *(model-agnostic block)*
`encode()` and `command_latent()` are the only functions a backbone port swaps.

In [ ]:
# ============================================================================
# MODEL LOAD + DCA HEADS (Cosmos3-Nano reasoner) -- v2
# Adds: 6-class mode head, 5-bin ordinal velocity head, frozen-z0 cache.
# ============================================================================
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import Cosmos3OmniForConditionalGeneration, AutoProcessor
from peft import LoraConfig, get_peft_model
from PIL import Image

VEL_EDGES = CFG["vel_edges"]
VEL_REPS  = CFG["vel_reps"]
ALL_LOSS_KEYS = ["vel", "dir", "mode", "obs_dist", "obs_close", "trav",
                 "unsafe", "false", "gate", "prior", "temp", "consist"]

processor = AutoProcessor.from_pretrained(CFG["base_model"])

def load_fresh_model(config=None):
    """Load a fresh Cosmos3-Nano reasoner + LoRA. Called once per (phase, rung)."""
    m = Cosmos3OmniForConditionalGeneration.from_pretrained(
        CFG["base_model"], dtype=torch.bfloat16, low_cpu_mem_usage=True)
    m = m.to("cuda:0")
    # v3: Apply config overrides
    rank = config.get("lora_rank_override") if config and config.get("lora_rank_override") else CFG["lora_rank"]
    dropout = config.get("lora_dropout") if config else CFG["lora_dropout"]
    m = get_peft_model(m, LoraConfig(
        r=rank, lora_alpha=rank*2, lora_dropout=dropout,
        target_modules="all-linear", init_lora_weights="gaussian"))
    if config and config.get("lora_rank_override"):
        print(f"  [v3] LoRA rank={rank}, dropout={dropout}")
    m.print_trainable_parameters()
    return m

def _hidden_size(cfg):
    for path in ("text_config.hidden_size", "hidden_size"):
        o = cfg
        for p in path.split("."):
            if hasattr(o, p): o = getattr(o, p)
            else: break
        else:
            if isinstance(o, int): return o
    return 4096

class DCAHeads(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.vel      = nn.Linear(d, 1)                 # continuous velocity (regression path)
        self.vel_ord  = nn.Linear(d, len(VEL_REPS))     # ordinal velocity over 5 bands
        self.dir      = nn.Linear(d, 2)
        self.mode     = nn.Linear(d, 4)                 # legacy 4-class
        self.mode6    = nn.Linear(d, 6)                 # v2: side-aware
        self.gate     = nn.Linear(d, 1)
        self.obs_dist = nn.Linear(d, 1)
        self.obs_close= nn.Linear(d, 1)
        self.trav     = nn.Linear(d, 1)
        nn.init.constant_(self.gate.bias, 2.0)          # gate open at init

def PROMPT(instr):
    return instr.strip()

def encode(image_path, instr):
    messages = [{"role": "user",
                 "content": [{"type": "image", "path": str(image_path)},
                             {"type": "text",  "text": PROMPT(instr)}]}]
    enc = processor.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_dict=True, return_tensors="pt")
    out = {}
    for k, v in enc.items():
        out[k] = v.to(DEV, dtype=torch.bfloat16) if k == "pixel_values" else v.to(DEV)
    return out

def command_latent(vla, item, use_base):
    ctx = vla.disable_adapter() if use_base else contextlib.nullcontext()
    kw = dict(input_ids=item["input_ids"], attention_mask=item["attention_mask"],
              pixel_values=item["pixel_values"], output_hidden_states=True, use_cache=False)
    for k in ("image_grid_thw", "mm_token_type_ids"):
        if k in item: kw[k] = item[k]
    with ctx:
        out = vla(**kw)
    return out.hidden_states[-1][:, -1, :].float()

# ============================================================================
# FROZEN-z0 CACHE
# z0 depends only on (image, instruction) and the FROZEN base weights -- identical across
# every epoch, every rung and every phase, yet v1 recomputed it on every single sample.
# Caching it removes one full backbone forward per sample. ~16 KB/frame on CPU.
# ============================================================================
Z0_CACHE = {}

def base_latent(vla, item, key):
    if FIX.get("cache_z0"):
        hit = Z0_CACHE.get(key)
        if hit is not None:
            return hit.to(DEV, non_blocking=True)
    z = command_latent(vla, item, use_base=True)
    if FIX.get("cache_z0"):
        Z0_CACHE[key] = z.detach().to("cpu")
    return z

@torch.no_grad()
def prebuild_z0(vla, clip_sets):
    """One pass over every frame to fill Z0_CACHE. Costs 1 forward/frame ONCE instead of
    1 forward/frame/epoch/rung."""
    if not FIX.get("cache_z0"): return
    todo = [(f["image"], c["instruction"]) for cs in clip_sets for c in cs for f in c["frames"]
            if f["image"] not in Z0_CACHE]
    if not todo:
        print(f"  z0 cache already warm ({len(Z0_CACHE)} frames)"); return
    print(f"  building z0 cache for {len(todo)} frames ...", flush=True)
    t0 = time.time()
    for i, (img, instr) in enumerate(todo):
        Z0_CACHE[img] = command_latent(vla, encode(img, instr), use_base=True).detach().to("cpu")
        if (i + 1) % 500 == 0:
            print(f"    z0 {i+1}/{len(todo)}  ({(time.time()-t0)/60:.1f} min)", flush=True)
    mb = sum(v.numel() * 4 for v in Z0_CACHE.values()) / 1e6
    print(f"  z0 cache ready: {len(Z0_CACHE)} frames, {mb:.0f} MB CPU, {(time.time()-t0)/60:.1f} min")

print("model factory + DCA heads v2 ready (mode6, 5-bin ordinal vel, z0 cache)")

## Part E — DCA Losses + Train/Eval
All loss terms are FIX-gated. `evaluate()` now returns every metric **alongside its
trivial-predictor reference**, computed from the ground truth of the same split.

In [ ]:
def cagrad_loss_combine(losses, params, alpha=0.4):
    from torch.autograd import grad
    active = {k: v for k, v in losses.items() if v is not None and v.item() > 1e-9}
    if len(active) <= 1:
        return sum(active.values()) if active else torch.tensor(0.0)
    
    grads = {}
    for name, loss_val in active.items():
        try:
            g = grad(loss_val, params, retain_graph=True, create_graph=False)
            g_flat = torch.cat([p.flatten() for p in g if p is not None])
            if g_flat.numel() > 0:
                grads[name] = g_flat
        except:
            continue
    
    if len(grads) == 0:
        return sum(active.values())
    
    g_avg = torch.stack(list(grads.values())).mean(0)
    dots = {k: torch.dot(g, g_avg).item() for k, g in grads.items()}
    worst = min(dots.keys(), key=lambda k: dots[k])
    
    weights = {k: (1.0 + alpha) if k == worst else 1.0 for k in active}
    total_w = sum(weights.values())
    weights = {k: v/total_w for k, v in weights.items()}
    
    return sum(weights[k] * active[k] for k in active)


In [ ]:
def compute_affordance_diagnostics(predictions):
    import numpy as np
    from collections import defaultdict
    if len(predictions) == 0:
        return {'collapse_detected': True, 'obs_dist_std': 0.0}
    
    obs_preds = [p['obs_dist_pred_m'] for p in predictions]
    obs_true = [p['obs_dist'] for p in predictions]
    
    obs_std = float(np.std(obs_preds))
    obs_corr = float(np.corrcoef(obs_preds, obs_true)[0,1]) if len(obs_preds) > 1 else 0.0
    obs_unique = len(set([round(p, 2) for p in obs_preds]))
    
    return {
        'obs_dist_std': obs_std,
        'obs_dist_corr': obs_corr,
        'obs_dist_unique': obs_unique,
        'collapse_detected': (obs_std < 0.6 or obs_unique < 100),
    }


In [ ]:
# ============================================================================
# DCA FORWARD + LOSSES + TRAIN/EVAL -- v2
# ============================================================================
import os, json, math, time, gc, re, random
from collections import Counter
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from sklearn.metrics import f1_score, roc_auc_score, r2_score

def cb_focal_loss(logits, target, cb_w, gamma):
    logp = F.log_softmax(logits, dim=-1)
    ce   = F.nll_loss(logp, target, weight=cb_w, reduction="none")
    pt   = logp.gather(1, target.view(-1, 1)).squeeze(1).exp()
    return ((1 - pt) ** gamma * ce).mean()

class UncertaintyWeights(nn.Module):
    """Kendall homoscedastic-uncertainty weighting: total = sum_k exp(-s_k) L_k + s_k."""
    def __init__(self, keys):
        super().__init__()
        self.log_sigma = nn.ParameterDict({k: nn.Parameter(torch.zeros(())) for k in keys})
    def combine(self, L):
        total = 0.0
        for k, v in L.items():
            if k in self.log_sigma:
                s = self.log_sigma[k]; total = total + torch.exp(-s) * v + s
            else:
                total = total + v
        return total

def commands(heads, adapted, base, flags, config):
    if flags["gate"] or flags["prior"]:
        delta = adapted - base.detach()
        gate  = torch.sigmoid(heads.gate(base.detach())) if flags["gate"] \
                else torch.ones(adapted.size(0), 1, device=adapted.device)
        hcmd  = base.detach() + gate * delta
    else:
        gate  = torch.sigmoid(heads.gate(adapted)); hcmd = adapted
    # v3: configurable affordance decoupling
    if config.get("afford_decouple", False) and base is not None:
        z_aff = base.detach()
    else:
        z_aff = adapted if FIX.get("afford_raw") else hcmd
    dvec  = heads.dir(hcmd); dvec = dvec / (dvec.norm(dim=-1, keepdim=True) + 1e-6)
    mlog  = heads.mode6(hcmd) if FIX.get("mode6") else heads.mode(hcmd)
    if FIX.get("ordinal_vel"):
        vlog = heads.vel_ord(hcmd)
        reps = torch.tensor(VEL_REPS, device=vlog.device, dtype=vlog.dtype)
        v    = (F.softmax(vlog, dim=-1) * reps).sum(-1)      # differentiable expected velocity
    else:
        vlog = None
        v    = F.softplus(heads.vel(hcmd)).squeeze(-1)
    return {"v": v, "dvec": dvec, "mlog": mlog, "gate": gate.squeeze(-1), "hcmd": hcmd,
            "odist": heads.obs_dist(z_aff).squeeze(-1),
            "oclose": heads.obs_close(z_aff).squeeze(-1),
            "trav": heads.trav(z_aff).squeeze(-1), "vlog": vlog}

def forward_frame(vla, heads, frame, instr, flags, config):
    it      = encode(frame["image"], instr)
    adapted = command_latent(vla, it, use_base=False)
    base    = base_latent(vla, it, frame["image"]) if (flags["gate"] or flags["prior"]) else None
    return commands(heads, adapted, base, flags, config), base

def make_tgt(f):
    th = math.radians(f["dir_deg"])
    t  = lambda x: torch.tensor([x], device=DEV, dtype=torch.float32)
    sev = 0.0
    if f["obs_valid"] > 0.5 and f["obs_close"] > 0.5:
        sev = max(0.0, min(1.0, 1.0 - f["obs_dist"] / CFG["close_m"]))
    mid = f["mode6_id"] if FIX.get("mode6") else f["mode_id"]
    return {"v": t(f["v"]), "cos": t(math.cos(th)), "sin": t(math.sin(th)),
            "absdir": t(abs(f["dir_deg"])),
            "mode_id": torch.tensor([mid], device=DEV),
            "rho": t(f["rho"]), "stop": t(f["stop"]),
            "obs_dist_n": t(f["obs_dist"] / CFG["obs_norm_m"]), "obs_close": t(f["obs_close"]),
            "obs_valid": t(f["obs_valid"]), "trav": t(f["trav"]), "trav_valid": t(f["trav_valid"]),
            "severity": t(sev)}

def per_frame_losses(config, out, base, tgt, flags):
    # v3: Apply config lambda overrides
    lam_obs_dist = config.get("lambda_obs_dist", lam_obs_dist)
    lam_obs_close = config.get("lambda_obs_close", lam_obs_close)
    lam_trav = config.get("lambda_trav", lam_trav)

        v, dvec, mlog, gate, hcmd = out["v"], out["dvec"], out["mlog"], out["gate"], out["hcmd"]
    odist, oclose, trav, vlog = out["odist"], out["oclose"], out["trav"], out["vlog"]
    L = {}
    v_t = tgt["v"]
    if FIX.get("safe_relabel"):
        v_t = v_t * (1.0 - CFG["relabel_max_slow"] * tgt["severity"])

    # --- velocity: ordinal classification (class-balanced in v2) or L1 regression ---
    if FIX.get("ordinal_vel"):
        edges = torch.tensor(VEL_EDGES, device=v.device)
        btgt  = torch.bucketize(v_t.detach(), edges).clamp(0, len(VEL_REPS) - 1)
        w     = CB_VEL_W.to(vlog.device) if FIX.get("cb_vel") else None
        L["vel"] = F.cross_entropy(vlog, btgt, weight=w) + 0.2 * (v - v_t).abs().mean()
    else:
        L["vel"] = (v - v_t).abs().mean()

    # --- direction: optionally reweighted toward informative (non-straight) frames ---
    cosd = tgt["cos"] * dvec[:, 0] + tgt["sin"] * dvec[:, 1]
    if FIX.get("dir_reweight"):
        fw = CFG["dir_floor_w"]
        wd = fw + (1.0 - fw) * (tgt["absdir"] / CFG["dir_ref_deg"]).clamp(0.0, 1.0)
        L["dir"] = (wd * (1 - cosd)).mean()
    else:
        L["dir"] = (1 - cosd).mean()

    # --- mode: class-balanced focal or weighted CE; 6-class when mode6 is on ---
    cbw = (CB_MODE6_W if FIX.get("mode6") else CB_MODE_W).to(mlog.device)
    iww = (MODE6_W    if FIX.get("mode6") else MODE_W   ).to(mlog.device)
    if FIX.get("cb_focal"):
        L["mode"] = cb_focal_loss(mlog, tgt["mode_id"], cbw, CFG["focal_gamma"])
    else:
        L["mode"] = F.cross_entropy(mlog, tgt["mode_id"], weight=iww)

    # --- affordance (always on, in every rung -> NOT an ablation axis) ---
    L["obs_dist"]  = (tgt["obs_valid"] * (odist - tgt["obs_dist_n"]).abs()).mean()
    L["obs_close"] = F.binary_cross_entropy_with_logits(oclose, tgt["obs_close"])
    L["trav"]      = (tgt["trav_valid"] * (trav - tgt["trav"]).abs()).mean()

    # --- v2: mode <-> velocity consistency -----------------------------------
    # Nothing in v1 tied "I predict stand" to "I command ~0 m/s". P2 dca_full said stand on
    # 45% of stop frames while commanding >= 0.5 m/s on 84% of close frames. This term makes
    # the two heads pay for disagreeing.
    if FIX.get("consist"):
        p_stand = F.softmax(mlog, dim=-1)[:, 0]
        L["consist"] = (p_stand * v).mean()

    # --- safety penalties -----------------------------------------------------
    if flags.get("unsafe"):
        oc = tgt["obs_close"]
        if FIX.get("unsafe_relative"):
            # Both terms measure against the demonstrator's own speed at THIS frame.
            # Absolute constants made L_false dead (relu(0.15 - v_pred) never fired) and
            # made L_unsafe demand a freeze the vel term simultaneously forbade.
            m  = CFG["rel_margin"]
            up = torch.clamp(v - (v_t + m), min=0)
            fp = torch.clamp((v_t - m) - v, min=0)
        else:
            up = torch.clamp(v - CFG["eps_stop"], min=0)
            fp = torch.clamp(CFG["crawl_speed"] - v, min=0)
        if not FIX.get("huber_pen"):
            up = up ** 2; fp = fp ** 2
        L["unsafe"] = (oc * up).mean()
        L["false"]  = ((1 - oc) * fp).mean()
    if flags.get("gate"):
        L["gate"] = F.binary_cross_entropy(gate.clamp(1e-6, 1 - 1e-6), tgt["rho"])
    if flags.get("prior"):
        L["prior"] = ((1 - tgt["rho"]) * ((hcmd - base.detach()) ** 2).mean(-1)).mean()
    return L

# ============================================================================
# SAMPLE STREAM
# v1 built (prev, cur) pairs and re-ran the backbone on `prev` every step -> 4 forwards per
# sample for dca_full. v2 walks each clip in order and reuses the PREVIOUS STEP's detached
# output as the temporal partner (it was already no_grad, so nothing is lost but one
# optimiser step of staleness). Clips are round-robin interleaved so a grad-accum window
# still mixes n_interleave different clips.
# ============================================================================
def build_stream(clips, flags, seed=0):
    use_temp = CFG["temporal_pairs"] and flags.get("temp", False)
    if not (use_temp and FIX.get("stream_pairs")):
        out = []
        for c in clips:
            fs = c["frames"]
            for i, f in enumerate(fs):
                prev = fs[i - 1] if (i > 0 and use_temp) else None
                out.append({"f": f, "prev": prev, "instr": c["instruction"],
                            "key": id(c), "start": i == 0})
        random.Random(seed).shuffle(out)
        return out, use_temp
    order = list(clips); random.Random(seed).shuffle(order)
    K = max(1, CFG["n_interleave"])
    out = []
    for g0 in range(0, len(order), K):
        grp   = order[g0:g0 + K]
        iters = [iter(list(enumerate(c["frames"]))) for c in grp]
        alive = list(range(len(grp)))
        while alive:
            nxt = []
            for j in alive:
                try: fi, f = next(iters[j])
                except StopIteration: continue
                out.append({"f": f, "prev": None, "instr": grp[j]["instruction"],
                            "key": (g0, j), "start": fi == 0})
                nxt.append(j)
            alive = nxt
    return out, use_temp

def train_step(config, vla, heads, item, flags, uw=None, memo=None):
    cur = item["f"]
    out, base = forward_frame(vla, heads, cur, item["instr"], flags, config)
    L = per_frame_losses(config, out, base, make_tgt(cur), flags)

    if flags.get("temp"):
        pv = None
        if FIX.get("stream_pairs"):
            if memo is not None and not item["start"]:
                pv = memo.get(item["key"])
        elif item["prev"] is not None:
            with torch.no_grad():
                po, _ = forward_frame(vla, heads, item["prev"], item["instr"], flags, config)
            pv = (po["v"].detach(), po["dvec"].detach(), item["prev"]["rho"])
        if pv is not None:
            p_v, p_d, p_rho = pv
            drho = abs(cur["rho"] - p_rho)
            dv   = (out["v"] - p_v).abs()
            dcos = 1 - (out["dvec"][:, 0] * p_d[:, 0] + out["dvec"][:, 1] * p_d[:, 1])
            L["temp"] = ((1 - drho) * (dv + CFG["temp_alpha"] * dcos)).mean()
        if memo is not None:
            memo[item["key"]] = (out["v"].detach(), out["dvec"].detach(), cur["rho"])

    # v3: CAGrad if enabled
    if config.get("use_cagrad", False):
        params = list(vla.parameters()) + list(heads.parameters())
        total = cagrad_loss_combine(L, params, alpha=0.4)
    elif FIX.get("uncert_weight") and uw is not None:
        total = uw.combine(L)
    else:
        LAM = CFG["lambda"]; total = sum(LAM.get(k, 1.0) * L[k] for k in L)
    total = total * cur.get("weight", 1.0)
    return total, {k: float(x.detach()) for k, x in L.items()}

# ============================================================================
# DECODE
# ============================================================================
def vec_to_label(mode_pred, v, cos_t, sin_t):
    """v2 (mode6): side comes from the classifier. v1 (mode4): side decoded from sin(dir),
    which is provably wrong for turns -- turning_right is 70% negative dir_deg."""
    if FIX.get("mode6"):
        name = MODE6_NAMES[mode_pred]
        if name == "stand":      return "standing_still"
        if name == "turn_left":  return "turning_left"
        if name == "turn_right": return "turning_right"
        if name == "step_left":  return "stepping_left"
        if name == "step_right": return "stepping_right"
    else:
        mode = {0: "stand", 1: "walk", 2: "turn", 3: "sidestep"}[mode_pred]
        if mode == "stand":    return "standing_still"
        if mode == "turn":     return "turning_left" if sin_t < 0 else "turning_right"
        if mode == "sidestep": return "stepping_left" if sin_t < 0 else "stepping_right"
    if v < BANDS["band_slow"]:   return "walking_slow"
    if v < BANDS["band_normal"]: return "walking_normal"
    return "walking_fast"

# ============================================================================
# TRIVIAL-PREDICTOR REFERENCES
# The single most important change in v2. Every headline number in P1/P2 was an absolute
# value with no reference, which is how a frozen model (scene_unsafe_go 0.043) read as the
# best safety result in the sweep when the human it imitates scores 0.675.
# ============================================================================
def _angle_err_deg(dir_deg, pred_deg):
    th, p = np.radians(dir_deg), math.radians(pred_deg)
    return np.degrees(np.arccos(np.clip(np.cos(th) * math.cos(p) + np.sin(th) * math.sin(p), -1, 1)))

def _corr(a, b):
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    if len(a) > 1 and np.std(a) > 1e-6 and np.std(b) > 1e-6:
        return float(np.corrcoef(a, b)[0, 1])
    return float("nan")

# ---- skill scores: 0 = no better than the trivial predictor, 1 = perfect ----
def _skill_max(val, trivial, ceiling):
    if not np.isfinite(val) or not np.isfinite(trivial) or ceiling <= trivial: return float("nan")
    return float(max(0.0, min(1.0, (val - trivial) / (ceiling - trivial))))

def _skill_min(val, trivial):
    if not np.isfinite(val) or not np.isfinite(trivial) or trivial <= 0: return float("nan")
    return float(max(0.0, min(1.0, 1.0 - val / trivial)))

def _skill_close(val, target):
    """For metrics where the DEMONSTRATOR is the target, not zero. Scoring jitter as
    'lower is better' would hand a perfect score to a frozen model -- the exact failure
    P1 was rewarded for. Closeness to the demonstrator is the honest objective."""
    if not np.isfinite(val) or not np.isfinite(target) or abs(target) < 1e-9: return float("nan")
    return float(max(0.0, min(1.0, 1.0 - abs(val - target) / abs(target))))

def trivial_references(yt, v_true, dir_true, obs_dist, obs_close, obs_valid, gt_jitter):
    v_true, dir_true = np.asarray(v_true), np.asarray(dir_true)
    obs_dist, obs_close = np.asarray(obs_dist), np.asarray(obs_close)
    obs_valid = np.asarray(obs_valid)
    ref = {}
    maj = Counter(yt).most_common(1)[0][0]
    ref["ref_macro_f1_trivial"] = float(f1_score(yt, [maj] * len(yt), labels=LABELS,
                                                 average="macro", zero_division=0))
    # ceiling reachable from a PERFECT mode head with velocity bands collapsed to the
    # majority walk label -- i.e. how much of macro_f1 is mode's job vs velocity's.
    walk_labels = [l for l in yt if l.startswith("walking")]
    maj_walk = Counter(walk_labels).most_common(1)[0][0] if walk_labels else "walking_normal"
    oracle = [maj_walk if l.startswith("walking") else l for l in yt]
    ref["ref_macro_f1_mode_oracle"] = float(f1_score(yt, oracle, labels=LABELS,
                                                     average="macro", zero_division=0))
    ref["ref_vel_mae_median"] = float(np.abs(v_true - np.median(v_true)).mean())
    ref["ref_dir_err_straight"] = float(_angle_err_deg(dir_true, 0.0).mean())
    ref["ref_jitter_demo"] = float(np.mean(gt_jitter)) if len(gt_jitter) else float("nan")
    ref["ref_turn_side_chance"] = 0.5
    cf = obs_close > 0.5
    ref["ref_scene_unsafe_go_demo"] = float((v_true[cf] >= CFG["go_speed"]).mean()) if cf.any() else float("nan")
    ov = obs_valid > 0.5
    if ov.sum() > 1 and np.std(v_true[ov]) > 1e-6 and np.std(obs_dist[ov]) > 1e-6:
        ref["ref_safe_speed_corr_demo"] = float(np.corrcoef(v_true[ov], obs_dist[ov])[0, 1])
    else:
        ref["ref_safe_speed_corr_demo"] = float("nan")
    ref["ref_tracks_demo"] = 1.0          # the demonstrator tracks itself by construction
    ref["ref_too_fast_demo"] = 0.0
    ref["ref_too_slow_demo"] = 0.0
    return ref

@torch.no_grad()
def evaluate(vla, heads, clips, tag, flags, dump_path=None, max_frames=None, quiet=False):
    vla.eval(); heads.eval()
    yt, yp, verr, derr, jit, gt_jit = [], [], [], [], [], []
    v_true, dir_true, od_all, oc_all, ov_all = [], [], [], [], []
    oc_true, oc_pred, od_true, od_pred, tr_true, tr_pred = [], [], [], [], [], []
    vcorr_p, vcorr_d = [], []
    mode_ok = mode_n = 0
    stop_t = stop_ok = open_t = false_stop = 0
    close_t = scene_unsafe = 0
    side_t = side_ok = 0
    cf_vp, cf_vt = [], []
    dump, seen = [], 0
    for ci, c in enumerate(clips):
        if max_frames and seen >= max_frames: break
        prev = prev_gt = None
        for f in c["frames"]:
            if max_frames and seen >= max_frames: break
            seen += 1
            out, _ = forward_frame(vla, heads, f, c["instruction"], flags)
            vv = float(out["v"][0]); cc = float(out["dvec"][0, 0]); ss = float(out["dvec"][0, 1])
            mode_pred = int(out["mlog"].argmax(dim=-1)[0])
            pl = vec_to_label(mode_pred, vv, cc, ss)
            mode_true = f["mode6_id"] if FIX.get("mode6") else f["mode_id"]
            mode_n += 1; mode_ok += int(mode_pred == mode_true)

            yt.append(f["label"]); yp.append(pl)
            verr.append(abs(f["v"] - vv))
            th = math.radians(f["dir_deg"])
            derr.append(math.degrees(math.acos(max(-1, min(1, math.cos(th)*cc + math.sin(th)*ss)))))
            v_true.append(f["v"]); dir_true.append(f["dir_deg"])
            od_all.append(f["obs_dist"]); oc_all.append(f["obs_close"]); ov_all.append(f["obs_valid"])

            gt_stop, pr_stop = f["label"] in STOP_LABELS, pl in STOP_LABELS
            if gt_stop: stop_t += 1; stop_ok += int(pr_stop)
            else:       open_t += 1; false_stop += int(pr_stop)

            if f["obs_valid"] > 0.5:
                oc_true.append(int(f["obs_close"])); oc_pred.append(float(torch.sigmoid(out["oclose"])[0]))
                od_true.append(f["obs_dist"]);       od_pred.append(float(out["odist"][0]) * CFG["obs_norm_m"])
                vcorr_p.append(vv);                  vcorr_d.append(f["obs_dist"])
            if f["trav_valid"] > 0.5:
                tr_true.append(f["trav"]); tr_pred.append(float(out["trav"][0]))
            if f["obs_close"] > 0.5:
                close_t += 1; scene_unsafe += int(vv >= CFG["go_speed"])
                cf_vp.append(vv); cf_vt.append(f["v"])

            # v2: turn-side accuracy over REAL turn/sidestep frames only. v1 measured it
            # over RARE, which included 215 walking_fast frames of the 314 total -- 68%
            # trivially-passing padding that inflated 0.55 into a reported 0.86.
            if f["label"] in TURN_STEP:
                side_t += 1
                side_ok += int(("left" in f["label"]) == ("left" in pl))

            if prev is not None:
                jit.append(abs(vv - prev[0]) + abs(cc - prev[1]) + abs(ss - prev[2]))
                gt_jit.append(abs(f["v"] - prev_gt[0]) + abs(math.cos(th) - prev_gt[1])
                              + abs(math.sin(th) - prev_gt[2]))
            prev, prev_gt = (vv, cc, ss), (f["v"], math.cos(th), math.sin(th))

            if dump_path:
                dump.append({"video_id": c.get("video_id"), "image": f["image"],
                             "label": f["label"], "label_pred": pl,
                             "v_true": f["v"], "v_pred": vv,
                             "dir_deg_true": f["dir_deg"], "cos_pred": cc, "sin_pred": ss,
                             "mode_true": mode_true, "mode_pred": mode_pred,
                             "obs_dist": f["obs_dist"], "obs_close": f["obs_close"],
                             "obs_valid": f["obs_valid"], "trav": f["trav"],
                             "obs_dist_pred_m": float(out["odist"][0]) * CFG["obs_norm_m"],
                             "obs_close_pred": float(torch.sigmoid(out["oclose"])[0]),
                             "trav_pred": float(out["trav"][0]),
                             "gate_pred": float(out["gate"][0]), "weight": f.get("weight", 1.0)})
        if not quiet:
            print(f"    eval clip {ci+1}/{len(clips)}", flush=True)

    def _auc(a, b): return float(roc_auc_score(a, b)) if len(set(a)) > 1 else float("nan")
    def _r2(a, b):  return float(r2_score(a, b)) if len(a) > 1 else float("nan")

    per_class = f1_score(yt, yp, labels=LABELS, average=None, zero_division=0)
    res = {
        "tag": tag,
        "macro_f1":    float(f1_score(yt, yp, labels=LABELS, average="macro", zero_division=0)),
        "per_class_f1": {l: float(s) for l, s in zip(LABELS, per_class)},
        "mode_acc":    mode_ok / max(1, mode_n),
        "stop_recall": stop_ok / max(1, stop_t),
        # unsafe_go == 1 - stop_recall EXACTLY. Kept for continuity with P0-P2 only; never
        # plot it against stop_recall (v1 fig 16 did, producing a straight line y = 1-x).
        "unsafe_go":   (stop_t - stop_ok) / max(1, stop_t),
        "false_stop":  false_stop / max(1, open_t),
        "vel_mae":     float(np.mean(verr)),
        "dir_err_deg": float(np.mean(derr)),
        "jitter":      float(np.mean(jit)) if jit else 0.0,
        "turn_side_acc": side_ok / max(1, side_t),
        "obstacle_auc":  _auc(oc_true, oc_pred),
        "proximity_mae": float(np.mean(np.abs(np.array(od_true) - np.array(od_pred)))) if od_true else float("nan"),
        "proximity_r2":  _r2(od_true, od_pred),
        "trav_r2":       _r2(tr_true, tr_pred),
        "scene_unsafe_go": scene_unsafe / max(1, close_t),
    }
    # corr(predicted speed, obstacle distance) on obs_valid frames. Reported against the
    # demonstrator's own value -- P1 scored 0.71-0.88 against a human 0.43, i.e. it coupled
    # speed to distance about twice as hard as the person did. That is paralysis, not safety.
    res["safe_speed_corr"] = _corr(vcorr_p, vcorr_d)

    # --- symmetric safety triple on close-obstacle frames (sums to 1.0) ---
    if cf_vp:
        d = np.array(cf_vp) - np.array(cf_vt); m = CFG["rel_margin"]
        res.update({"too_fast": float((d >  m).mean()),
                    "too_slow": float((d < -m).mean()),
                    "tracks":   float((np.abs(d) <= m).mean()),
                    "close_v_bias": float(d.mean())})
    else:
        res.update({"too_fast": float("nan"), "too_slow": float("nan"),
                    "tracks": float("nan"), "close_v_bias": float("nan")})

    res.update(trivial_references(yt, v_true, dir_true, od_all, oc_all, ov_all, gt_jit))
    res.update({"n_frames": len(yt), "n_stop_frames": stop_t, "n_close_frames": close_t,
                "n_turn_step_frames": side_t, "n_clips": len(clips),
                "rel_margin_m": CFG["rel_margin"], "go_speed": CFG["go_speed"]})

    res["skill_macro_f1"] = _skill_max(res["macro_f1"], res["ref_macro_f1_trivial"], 1.0)
    res["skill_vel_mae"]  = _skill_min(res["vel_mae"], res["ref_vel_mae_median"])
    res["skill_dir_err"]  = _skill_min(res["dir_err_deg"], res["ref_dir_err_straight"])
    res["skill_jitter"]   = _skill_close(res["jitter"], res["ref_jitter_demo"])

    if dump_path:
        json.dump(dump, open(dump_path, "w"))
        if not quiet: print(f"    per-frame predictions -> {dump_path} ({len(dump)} frames)", flush=True)
    if not quiet:
        _short = {k: (round(v, 4) if isinstance(v, float) else v)
                  for k, v in res.items() if not isinstance(v, dict)}
        print(f"  {tag}: {json.dumps(_short)}")
    return res

print("DCA v2 losses + stream + evaluate ready (baseline-referenced metrics)")

## CPU SELF-TEST
Exercises every phase × rung loss path, both decode paths, the stream builder and the skill
scores on dummy tensors. Runs in <1s with no GPU. **Run this before renting compute.**

In [ ]:
# ============================================================================
# CPU SELF-TEST
# ============================================================================
def _selftest():
    global encode, command_latent, DEV, MODE_W, CB_MODE_W, MODE6_W, CB_MODE6_W, CB_VEL_W
    _enc, _cl, _dev = encode, command_latent, DEV
    DEV = "cpu"
    for nm, n in [("MODE_W", 4), ("CB_MODE_W", 4), ("MODE6_W", 6), ("CB_MODE6_W", 6),
                  ("CB_VEL_W", len(VEL_REPS))]:
        if nm not in globals(): globals()[nm] = torch.ones(n)
    def encode(img, instr): return {"_": True}
    def command_latent(vla, item, use_base): return torch.randn(1, 32)
    heads_t = DCAHeads(32).float()
    fake = {"image": "x.jpg", "label": "walking_normal", "v": 0.8, "dir_deg": 12.0,
            "mode_id": 1, "mode6_id": 1, "rho": 1.0, "stop": 0.0, "obs_dist": 1.2,
            "obs_close": 1.0, "obs_valid": 1.0, "trav": 0.3, "trav_valid": 1.0, "weight": 1.0}
    ok = True
    for pname, fx in PHASE_CONFIGS.items():
        FIX.clear(); FIX.update(fx); FIX["cache_z0"] = False   # no real latents on CPU
        for rname, fl in ABLATION_FLAGS.items():
            uw = UncertaintyWeights(ALL_LOSS_KEYS) if FIX.get("uncert_weight") else None
            try:
                memo = {}
                item = {"f": fake, "prev": fake, "instr": "go", "key": 0, "start": True}
                tot, parts = train_step(None, heads_t, item, fl, uw, memo)
                assert math.isfinite(float(tot.detach())), "non-finite total"
                tot.backward()
                item2 = dict(item, start=False)
                tot2, _ = train_step(None, heads_t, item2, fl, uw, memo)   # exercises temp path
                assert math.isfinite(float(tot2.detach())), "non-finite total (temp)"
                tot2.backward()
                n_modes = 6 if FIX.get("mode6") else 4
                for mp in range(n_modes):
                    lab = vec_to_label(mp, 0.7, 0.9, -0.4)
                    assert lab in LABELS_SELFTEST, f"decoded unknown label {lab}"
            except Exception as e:
                ok = False; print(f"  [FAIL] {pname:16s}/{rname:16s}: {type(e).__name__}: {e}")
    # stream builder
    FIX.clear(); FIX.update(PHASE_CONFIGS["V2_full"])
    fake_clips = [{"video_id": f"v{i}", "instruction": "go",
                   "frames": [dict(fake, image=f"v{i}_{j}.jpg") for j in range(5)]} for i in range(20)]
    st, ut = build_stream(fake_clips, ABLATION_FLAGS["dca_full"])
    assert len(st) == 100, f"stream lost frames: {len(st)}"
    assert sum(1 for s in st if s["start"]) == 20, "wrong number of clip starts"
    ks = [s["key"] for s in st[:CFG['n_interleave']]]
    assert len(set(ks)) == len(ks), "grad-accum window does not mix clips"
    # skill scores
    assert _skill_min(0.285, 0.308) > 0 and _skill_min(0.40, 0.308) == 0.0, "skill_min wrong"
    assert _skill_max(0.22, 0.084, 1.0) > 0 and _skill_max(0.05, 0.084, 1.0) == 0.0, "skill_max wrong"
    FIX.clear()
    encode, command_latent, DEV = _enc, _cl, _dev
    if ok: print("SELF-TEST: PASS (all phase x rung paths finite + differentiable; stream + skill OK)")
    else:  raise SystemExit("SELF-TEST FAILED -- fix the code before spending GPU time.")

LABELS_SELFTEST = set(LABELS) if "LABELS" in globals() else {
    "standing_still","walking_slow","walking_normal","walking_fast",
    "turning_left","turning_right","stepping_left","stepping_right"}
_selftest()

## Part F — Run the ladder
Fresh model per (phase, rung) → z₀ cache → train with per-epoch validation → best checkpoint
→ eval on test → free GPU → next.

In [ ]:
# ============================================================================
# DCA v2 SWEEP: for each PHASE x each RUNG -> train + validate + test
# Resumable: existing metrics_<rung>.json under out/<phase>/ are skipped.
# ============================================================================
from collections import defaultdict
from transformers import get_cosine_schedule_with_warmup
import copy

SWEEP_RESULTS, ALL_RESULTS, ALL_EPOCH_LOGS = {}, {}, {}

def _better(new, old, mode):
    if old is None or not np.isfinite(old): return True
    if not np.isfinite(new): return False
    return new > old if mode == "max" else new < old

for phase in PHASES_TO_RUN:
    FIX.clear(); FIX.update(PHASE_CONFIGS[phase])
    current_config = TEST_CONFIGS.get(phase, TEST_CONFIGS["baseline"])
    phase_dir = os.path.join(CFG["out"], phase); os.makedirs(phase_dir, exist_ok=True)
    print(f"\n{'#'*72}\nPHASE {phase} | fixes: {dict(FIX)}\n{'#'*72}")
    SWEEP_RESULTS[phase] = {}; phase_logs = {}

    for rung in RUNGS_TO_RUN:
        flags    = ABLATION_FLAGS[rung]
        rung_dir = os.path.join(phase_dir, rung); os.makedirs(rung_dir, exist_ok=True)
        mpath    = os.path.join(phase_dir, f"metrics_{rung}.json")
        if os.path.exists(mpath):
            SWEEP_RESULTS[phase][rung] = json.load(open(mpath))
            lp = os.path.join(rung_dir, "epoch_log.json")
            phase_logs[rung] = json.load(open(lp)) if os.path.exists(lp) else []
            print(f"  [resume] {phase}/{rung} present -> skip"); continue

        print(f"\n{'='*64}\n{phase} | RUNG {rung} | flags {flags}\n{'='*64}")
        vla   = load_fresh_model(current_config)
        D     = _hidden_size(vla.config)
        heads = DCAHeads(D).to("cuda:0").float()
        uw    = UncertaintyWeights(ALL_LOSS_KEYS).to("cuda:0") if FIX.get("uncert_weight") else None

        # z0 is identical across rungs and phases -> built once, reused for the whole sweep
        if flags["gate"] or flags["prior"]:
            prebuild_z0(vla, [train_clips, val_clips, test_clips])

        # fixed val subset, chosen once so the per-epoch curve is comparable across epochs
        # (slicing inside evaluate() would always take the same PREFIX of clips, biasing it)
        val_eval_clips = val_clips
        if val_clips and CFG["val_max_frames"]:
            _vv = val_clips[:]; random.Random(7).shuffle(_vv)
            val_eval_clips, _tot = [], 0
            for _c in _vv:
                if _tot >= CFG["val_max_frames"]: break
                val_eval_clips.append(_c); _tot += len(_c["frames"])

        stream, use_temp = build_stream(train_clips, flags, seed=CFG["split_seed"])
        print(f"  samples: {len(stream)} | temporal pairs: {use_temp} | "
              f"stream_pairs: {bool(FIX.get('stream_pairs'))} | z0 cached: {len(Z0_CACHE)}")

        params = [p for p in vla.parameters() if p.requires_grad] + list(heads.parameters())
        if uw is not None: params += list(uw.parameters())
        lr  = CFG["lr_stable"] if FIX.get("lr_sched") else CFG["lr"]
        opt = AdamW(params, lr=lr)
        sched = None
        if FIX.get("lr_sched"):
            tot_steps = CFG["epochs"] * max(1, len(stream) // CFG["grad_accum"])
            sched = get_cosine_schedule_with_warmup(opt, int(CFG["warmup_frac"] * tot_steps), tot_steps)

        # one-step smoke
        vla.train(); heads.train(); opt.zero_grad()
        _t1, _p1 = train_step(current_config, vla, heads, stream[0], flags, uw, {})
        _t1.backward()
        if FIX.get("grad_clip"): torch.nn.utils.clip_grad_norm_(params, CFG["grad_clip"])
        opt.step()
        if sched: sched.step()
        opt.zero_grad()
        assert math.isfinite(float(_t1.detach())), f"non-finite smoke loss {phase}/{rung}"
        print(f"  SMOKE OK | loss {float(_t1.detach()):.4f} | {_p1}")

        epoch_log, t0 = [], time.time()
        best_score, best_epoch = None, -1
        best_heads = None
        for ep in range(CFG["epochs"]):
            if FIX.get("epoch_flush"): opt.zero_grad()
            vla.train(); heads.train()
            run = defaultdict(float); seen = 0
            memo = {}                      # clip_key -> previous detached (v, dvec, rho)
            for i, item in enumerate(stream):
                total, parts = train_step(current_config, vla, heads, item, flags, uw, memo)
                (total / CFG["grad_accum"]).backward()
                for k, x in parts.items(): run[k] += x
                run["total"] += float(total.detach()); seen += 1
                if (i + 1) % CFG["grad_accum"] == 0:
                    if FIX.get("grad_clip"): torch.nn.utils.clip_grad_norm_(params, CFG["grad_clip"])
                    opt.step()
                    if sched: sched.step()
                    opt.zero_grad()
                if (i + 1) % max(1, len(stream)//4) == 0:
                    mem = torch.cuda.max_memory_allocated() / 1e9
                    shown = " ".join(f"{k}:{run[k]/seen:.3f}" for k in
                        ["total","vel","dir","mode","consist","obs_close","unsafe","false","gate","prior","temp"]
                        if k in run)
                    print(f"  {phase}/{rung} ep{ep+1}/{CFG['epochs']} {i+1}/{len(stream)} {shown} mem{mem:.1f}G", flush=True)
            avg = {k: round(run[k]/seen, 4) for k in run}

            # ---- per-epoch validation + best-checkpoint selection ----------------
            # v1 had no val set at all: fixed 10 epochs, one eval at the end, no way to see
            # that train CE reached ~0.22 while test vel_mae barely beat a constant.
            vmet = {}
            if val_eval_clips and CFG["val_every"] and (ep + 1) % CFG["val_every"] == 0:
                vres = evaluate(vla, heads, val_eval_clips, f"VAL[{phase}/{rung}] ep{ep+1}",
                                flags, quiet=True)
                vmet = {f"val_{k}": v for k, v in vres.items() if isinstance(v, (int, float))}
                score = vres.get(CFG["select_metric"])
                if _better(score, best_score, CFG["select_mode"]):
                    best_score, best_epoch = score, ep + 1
                    best_heads = copy.deepcopy(heads.state_dict())
                    vla.save_pretrained(f"{rung_dir}/adapter_best")
                    torch.save(best_heads, f"{rung_dir}/heads_best.pt")
                print(f"     val {CFG['select_metric']}={score:.4f} | best={best_score:.4f} @ep{best_epoch}"
                      f" | val_vel_mae={vres['vel_mae']:.3f} (ref {vres['ref_vel_mae_median']:.3f})"
                      f" | val_dir_err={vres['dir_err_deg']:.2f} (ref {vres['ref_dir_err_straight']:.2f})",
                      flush=True)
            epoch_log.append({"epoch": ep + 1, **avg, **vmet})
            print(f"  == {phase}/{rung} epoch {ep+1}: " + " ".join(f"{k}:{v}" for k, v in avg.items()))

        print(f"  trained in {(time.time()-t0)/60:.1f} min")
        vla.save_pretrained(f"{rung_dir}/adapter_final")
        torch.save(heads.state_dict(), f"{rung_dir}/heads_final.pt")
        if uw is not None: torch.save(uw.state_dict(), f"{rung_dir}/uncertainty.pt")

        # score the SELECTED checkpoint, not blindly the last one
        if best_heads is not None and best_epoch != CFG["epochs"]:
            print(f"  restoring best checkpoint (epoch {best_epoch}, val {CFG['select_metric']}={best_score:.4f})")
            heads.load_state_dict(best_heads)
            try:
                vla.load_adapter(f"{rung_dir}/adapter_best", adapter_name="default")
            except Exception as e:
                print(f"  [warn] could not reload best adapter ({e}); heads restored, adapter is final-epoch")

        res = evaluate(vla, heads, test_clips, f"{phase}[{rung}]", flags,
                       dump_path=os.path.join(CFG["out"],
                           "predictions_" + re.sub(r"[^A-Za-z0-9_.-]", "_", f"{phase}_{rung}") + ".json"))
        res["selected_epoch"] = best_epoch if best_heads is not None else CFG["epochs"]
        res["val_select_score"] = best_score
        json.dump(res, open(mpath, "w"), indent=2)
        json.dump(epoch_log, open(f"{rung_dir}/epoch_log.json", "w"), indent=2)
        SWEEP_RESULTS[phase][rung] = res; phase_logs[rung] = epoch_log

        del vla, heads, opt, best_heads
        if uw is not None: del uw
        gc.collect(); torch.cuda.empty_cache()
        print(f"  GPU freed.\n")

    ALL_RESULTS = SWEEP_RESULTS[phase]; ALL_EPOCH_LOGS = phase_logs

json.dump(SWEEP_RESULTS, open(os.path.join(CFG["out"], "sweep_results.json"), "w"), indent=2)
print("\nSWEEP COMPLETE. Phases:", list(SWEEP_RESULTS.keys()))

In [ ]:
# ============================================================================
# PHASE x RUNG COMPARISON -- every metric printed next to its trivial reference
# ============================================================================
METRICS = [
    ("macro_f1", "ref_macro_f1_trivial"), ("skill_macro_f1", None), ("mode_acc", None),
    ("stop_recall", None), ("false_stop", None),
    ("vel_mae", "ref_vel_mae_median"), ("skill_vel_mae", None),
    ("dir_err_deg", "ref_dir_err_straight"), ("skill_dir_err", None),
    ("turn_side_acc", "ref_turn_side_chance"),
    ("tracks", "ref_tracks_demo"), ("too_fast", None), ("too_slow", None), ("close_v_bias", None),
    ("jitter", "ref_jitter_demo"),
    ("scene_unsafe_go", "ref_scene_unsafe_go_demo"),
    ("safe_speed_corr", "ref_safe_speed_corr_demo"),
    ("obstacle_auc", None), ("proximity_r2", None), ("trav_r2", None),
    ("selected_epoch", None), ("n_stop_frames", None),
]
comp = dict(SWEEP_RESULTS)
for m, refk in METRICS:
    print(f"\n### {m}")
    print(f'{"phase":18s} | ' + " | ".join(f"{r[:14]:>14s}" for r in CFG["rungs"]) + " |  reference")
    print("-" * (34 + 17 * len(CFG["rungs"])))
    for ph, rr in comp.items():
        row, ref = [], None
        for r in CFG["rungs"]:
            v = rr.get(r, {}).get(m)
            row.append(f"{v:14.4f}" if isinstance(v, (int, float)) else f'{"-":>14s}')
            if refk and isinstance(rr.get(r, {}).get(refk), (int, float)): ref = rr[r][refk]
        rs = f"{ref:.4f}" if ref is not None else "-"
        print(f"{ph:18s} | " + " | ".join(row) + f" |  {rs}")

json.dump(comp, open(os.path.join(CFG["out"], "phase_rung_comparison.json"), "w"), indent=2)
print("\nsaved -> phase_rung_comparison.json")

## Acceptance Gate
v1's gate only checked two **saturated, rung-invariant** affordance metrics, so it printed
PASS for a model whose direction head was worse than a constant. v2 checks skill against the
trivial predictors and checks that the model has not simply frozen.

In [ ]:
# ============================================================================
# ACCEPTANCE GATE v2 -- baseline-relative, run on the saved metrics (no retraining)
# ============================================================================
PASS_OBS_AUC   = 0.80
PASS_PROX_R2   = 0.20
PASS_TOO_SLOW  = 0.50    # above this the model is in the paralysis regime P1 fell into

def gate_report(results):
    allok = True
    for rung, res in results.items():
        if not res: continue
        checks = [
            ("obstacle_auc >= 0.80",            res.get("obstacle_auc", float("nan")) >= PASS_OBS_AUC,
             f"{res.get('obstacle_auc', float('nan')):.3f}"),
            ("proximity_r2 >= 0.20",            res.get("proximity_r2", float("nan")) >= PASS_PROX_R2,
             f"{res.get('proximity_r2', float('nan')):.3f}"),
            ("macro_f1 > trivial",              res.get("macro_f1", 0) > res.get("ref_macro_f1_trivial", 1),
             f"{res.get('macro_f1', 0):.3f} vs {res.get('ref_macro_f1_trivial', float('nan')):.3f}"),
            ("vel_mae < predict-median",        res.get("vel_mae", 9) < res.get("ref_vel_mae_median", 0),
             f"{res.get('vel_mae', 9):.3f} vs {res.get('ref_vel_mae_median', float('nan')):.3f}"),
            ("dir_err < straight-ahead",        res.get("dir_err_deg", 999) < res.get("ref_dir_err_straight", 0),
             f"{res.get('dir_err_deg', 999):.2f} vs {res.get('ref_dir_err_straight', float('nan')):.2f}"),
            ("turn_side_acc > chance",          res.get("turn_side_acc", 0) > 0.5,
             f"{res.get('turn_side_acc', 0):.3f}"),
            ("not paralysed (too_slow < 0.5)",  res.get("too_slow", 1) < PASS_TOO_SLOW,
             f"{res.get('too_slow', float('nan')):.3f}"),
        ]
        print(f"================ {rung} ================")
        for name, ok, val in checks:
            print(f"  [{'PASS' if ok else 'FAIL'}] {name:32s}  {val}")
        rung_ok = all(c[1] for c in checks)
        allok = allok and rung_ok
        print(f"  n_frames={res.get('n_frames')} n_stop={res.get('n_stop_frames')} "
              f"n_close={res.get('n_close_frames')} n_turn_step={res.get('n_turn_step_frames')} "
              f"| selected epoch {res.get('selected_epoch')}")
        sr, n = res.get("stop_recall", float("nan")), max(1, res.get("n_stop_frames", 1))
        ci = 1.96 * math.sqrt(max(sr * (1 - sr), 1e-9) / n)
        print(f"  stop_recall {sr:.3f} +/- {ci:.3f} (95% CI, n={n}; correlated frames -> true CI is wider)")
        print(f"  rung verdict: {'PASS' if rung_ok else 'FAIL'}\n")
    print("=" * 52)
    print(f"ACCEPTANCE GATE: {'PASS' if allok else 'FAIL - see checks above'}")
    print("=" * 52)
    return allok

gate_report(ALL_RESULTS)

## Part G — Figures
Rebuilt. Every panel that has a trivial predictor draws it as a dashed reference line, the
radar plots **skill** (0 = no better than trivial) instead of raw values on mismatched scales,
unrun rungs are skipped rather than drawn as `0.000`, and a render smoke test fails the cell
if any figure's aspect ratio blows up again.

In [ ]:
# ============================================================================
# FIGURES v2
# ============================================================================
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image as PILImage

plt.rcParams.update({"figure.facecolor":"white","axes.facecolor":"white","axes.grid":True,
    "grid.alpha":0.3,"font.size":12,"axes.titlesize":14,"axes.labelsize":12,
    "figure.dpi":150,"savefig.bbox":"tight","savefig.dpi":200})

COLORS = {"total":"#1D9E75","vel":"#534AB7","dir":"#D85A30","mode":"#378ADD","consist":"#7E57C2",
          "unsafe":"#E24B4A","false":"#BA7517","gate":"#D4537E","prior":"#888780","temp":"#639922",
          "obs_dist":"#4DB6AC","obs_close":"#26A69A","trav":"#9CCC65"}
RUNG_COLORS = {"lora_only":"#888780","lora_unsafe":"#D85A30",
               "lora_unsafe_prior":"#534AB7","dca_full":"#1D9E75"}
RUNG_SHORT  = {"lora_only":"V0:LoRA","lora_unsafe":"V1:+Unsafe",
               "lora_unsafe_prior":"V2:+Prior","dca_full":"V3:DCA Full"}
LOSS_KEYS = ["vel","dir","mode","consist","unsafe","false","gate","prior","temp",
             "obs_dist","obs_close","trav"]

# (key, label, better, trivial-ref key, ceiling-ref key)
FIG_METRICS = [
    ("macro_f1",      "Macro-F1 ↑",        "max", "ref_macro_f1_trivial",  "ref_macro_f1_mode_oracle"),
    ("stop_recall",   "Stop-Recall ↑",     "max", None,                    None),
    ("vel_mae",       "Vel-MAE m/s ↓",     "min", "ref_vel_mae_median",    None),
    ("dir_err_deg",   "Dir-Err deg ↓",     "min", "ref_dir_err_straight",  None),
    ("tracks",        "Tracks-Demo ↑",     "max", None,                    "ref_tracks_demo"),
    # jitter is scored as CLOSENESS to the demonstrator, not "lower is better" -- a frozen
    # model has jitter 0 and would otherwise score perfectly on it.
    ("jitter",        "Jitter vs demo ↔",  "close", "ref_jitter_demo",     None),
]
AFFORD_METRICS = [("obstacle_auc","Obstacle-AUC ↑"), ("proximity_r2","Proximity-R2 ↑"),
                  ("trav_r2","Trav-R2 ↑")]

def _present(results):
    """Only rungs that actually ran. v1 defaulted missing rungs to 0 and drew a 0.000 bar,
    which reads as 'scored zero' rather than 'not run'."""
    return [r for r in CFG["rungs"] if results.get(r)]

def _skill_of(res, key, label, better, refk, ceilk):
    v = res.get(key)
    if v is None or not np.isfinite(v): return float("nan")
    if better == "min":
        return _skill_min(v, res.get(refk, float("nan")))
    if better == "close":
        return _skill_close(v, res.get(refk, float("nan")))
    ceiling = res.get(ceilk) if ceilk else 1.0
    if ceiling is None or not np.isfinite(ceiling): ceiling = 1.0
    return _skill_max(v, res.get(refk, 0.0) if refk else 0.0, ceiling)

def _txt_in(ax, x, y, s, **kw):
    """Place a label but never outside the axes -- v1's fig 5 wrote text at y=19.4 inside
    ylim (0, 1.15); with savefig bbox='tight' matplotlib grew the CANVAS to include it,
    which is what produced the 20:1 aspect-ratio image."""
    lo, hi = ax.get_ylim()
    ax.text(x, min(max(y, lo + 0.02 * (hi - lo)), hi - 0.06 * (hi - lo)), s, **kw)

def make_figures(results, epoch_logs, fig_dir, label_dist=None):
    os.makedirs(fig_dir, exist_ok=True)
    rungs = _present(results)
    made  = []
    ref   = results[rungs[0]] if rungs else {}

    def _save(fig, name):
        p = os.path.join(fig_dir, name); fig.savefig(p); plt.close(fig); made.append(p)

    # ── 01 total loss ──
    if epoch_logs:
        fig, ax = plt.subplots(figsize=(10, 6))
        for r, el in epoch_logs.items():
            if not el: continue
            ax.plot([d["epoch"] for d in el], [d["total"] for d in el], "o-",
                    color=RUNG_COLORS.get(r, "#333"), lw=2, ms=6, label=RUNG_SHORT.get(r, r))
        ax.set_xlabel("Epoch"); ax.set_ylabel("Loss"); ax.set_title("Total Training Loss — All Rungs")
        ax.legend(); _save(fig, "01_total_loss_all_rungs.png")

    # ── 02 loss components (LOG y) ──
    # v1 put L_vel (1.19) and L_prior (0.085) on one linear axis, so a 50% drop in the small
    # terms was invisible and read as "not learning".
    if epoch_logs.get("dca_full"):
        el  = epoch_logs["dca_full"]; eps = [d["epoch"] for d in el]
        fig, axs = plt.subplots(1, 2, figsize=(16, 6))
        for k in LOSS_KEYS:
            vals = [d.get(k, 0) for d in el]
            if not any(v > 0 for v in vals): continue
            axs[0].plot(eps, vals, "o-", color=COLORS.get(k, "#333"), lw=1.8, ms=5, label=f"L_{k}")
            v0 = vals[0] if vals[0] > 0 else 1.0
            axs[1].plot(eps, [v / v0 for v in vals], "o-", color=COLORS.get(k, "#333"), lw=1.8, ms=5)
        axs[0].set_yscale("log"); axs[0].set_title("DCA Full — Loss Components (log scale)")
        axs[0].set_xlabel("Epoch"); axs[0].set_ylabel("Loss"); axs[0].legend(ncol=2, fontsize=9)
        axs[1].axhline(1.0, color="black", lw=0.8, ls="--")
        axs[1].set_title("Same, normalised to epoch 1 (< 1 = improving)")
        axs[1].set_xlabel("Epoch"); axs[1].set_ylabel("L / L(epoch 1)")
        _save(fig, "02_loss_components_dca_full.png")

    # ── 03 stacked ──
    if epoch_logs.get("dca_full"):
        el = epoch_logs["dca_full"]; eps = [d["epoch"] for d in el]
        act = [k for k in LOSS_KEYS if any(d.get(k, 0) > 0 for d in el)]
        fig, ax = plt.subplots(figsize=(10, 6))
        ax.stackplot(eps, np.array([[d.get(k, 0) for d in el] for k in act]),
                     labels=[f"L_{k}" for k in act],
                     colors=[COLORS.get(k, "#aaa") for k in act], alpha=0.8)
        ax.set_xlabel("Epoch"); ax.set_ylabel("Cumulative Loss")
        ax.set_title("DCA Full — Loss Composition"); ax.legend(loc="upper right", ncol=2, fontsize=9)
        _save(fig, "03_loss_stacked.png")

    # ── 04 radar: SKILL, not raw values ──
    # v1 plotted raw values on a 0-1 polar axis, so Dir-Err (18.5) shot off the chart and
    # Scene-Unsafe-Go 0.837 (bad) rendered as a big radius (good). Skill is unit-free,
    # bounded, and orientation-correct: 0 = no better than the trivial predictor.
    for r in rungs:
        res  = results[r]
        sk   = [_skill_of(res, *m) for m in FIG_METRICS]
        sk   = [0.0 if not np.isfinite(s) else s for s in sk]
        raw  = [res.get(m[0], float("nan")) for m in FIG_METRICS]
        N    = len(FIG_METRICS)
        ang  = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
        fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
        ax.plot(ang + ang[:1], sk + sk[:1], "o-", color=RUNG_COLORS.get(r, "#333"), lw=2)
        ax.fill(ang + ang[:1], sk + sk[:1], alpha=0.2, color=RUNG_COLORS.get(r, "#333"))
        ax.set_xticks(ang); ax.set_xticklabels([m[1] for m in FIG_METRICS], fontsize=10)
        ax.set_ylim(0, 1); ax.set_yticks([0.25, 0.5, 0.75, 1.0])
        for a, s, rv, m in zip(ang, sk, raw, FIG_METRICS):
            ax.annotate(f"{rv:.3g}\n(skill {s:.2f})", xy=(a, min(s + 0.06, 0.94)),
                        fontsize=8, ha="center", fontweight="bold")
        ax.set_title(f"{r} — skill vs. trivial baseline\n0 = no better than trivial, 1 = perfect",
                     pad=24, fontsize=13)
        _save(fig, f"04_radar_{r}.png")

    # ── 05 grouped bars of SKILL (all bounded 0-1, so the scales finally match) ──
    fig, ax = plt.subplots(figsize=(14, 6))
    x = np.arange(len(FIG_METRICS)); w = 0.8 / max(1, len(rungs))
    ax.set_ylim(0, 1.18)
    for i, r in enumerate(rungs):
        sk  = [_skill_of(results[r], *m) for m in FIG_METRICS]
        sk  = [0.0 if not np.isfinite(s) else s for s in sk]
        raw = [results[r].get(m[0], float("nan")) for m in FIG_METRICS]
        bars = ax.bar(x + i*w, sk, w, label=RUNG_SHORT.get(r, r),
                      color=RUNG_COLORS.get(r, "#888"), alpha=0.88, edgecolor="black")
        for b, s, rv in zip(bars, sk, raw):
            _txt_in(ax, b.get_x() + b.get_width()/2, s + 0.02, f"{rv:.3g}",
                    ha="center", va="bottom", fontsize=8, fontweight="bold")
    ax.axhline(0.0, color="#E24B4A", lw=1.4, ls="--", label="trivial baseline (skill 0)")
    ax.set_ylabel("Skill (0 = trivial predictor, 1 = perfect)")
    ax.set_title("Ablation Comparison — skill vs. trivial baseline (raw value printed on each bar)")
    ax.set_xticks(x + w*(len(rungs)-1)/2); ax.set_xticklabels([m[1] for m in FIG_METRICS])
    ax.legend(); _save(fig, "05_ablation_bars.png")

    # ── 06 REAL safety trade-off: the symmetric triple ──
    # v1 plotted unsafe_go vs false_stop; false_stop's whole range was 0.0009-0.017 on a 0-1
    # axis, so every point sat on the floor and the 0.5 quadrant lines were unreachable.
    fig, ax = plt.subplots(figsize=(10, 6))
    names = ["demonstrator"] + [RUNG_SHORT.get(r, r) for r in rungs]
    slow  = [0.0] + [results[r].get("too_slow", np.nan) for r in rungs]
    trk   = [1.0] + [results[r].get("tracks",   np.nan) for r in rungs]
    fast  = [0.0] + [results[r].get("too_fast", np.nan) for r in rungs]
    y = np.arange(len(names))
    ax.barh(y, slow, color="#BA7517", label="too slow (paralysis)")
    ax.barh(y, trk,  left=slow, color="#1D9E75", label="tracks demonstrator")
    ax.barh(y, fast, left=np.array(slow)+np.array(trk), color="#E24B4A", label="too fast")
    ax.set_yticks(y); ax.set_yticklabels(names); ax.invert_yaxis()
    ax.set_xlabel(f"fraction of close-obstacle frames (deadband +/- {CFG['rel_margin']} m/s)")
    ax.set_title("Safety Trade-off — symmetric, referenced to the demonstrator\n"
                 "(paralysis shows up as 'too slow'; the one-sided legacy metric scored it as perfect)")
    # bars always span the full width (the triple sums to 1), so the legend goes outside
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.16), ncol=3, frameon=False)
    ax.set_xlim(0, 1); _save(fig, "06_safety_tradeoff.png")

    # ── 07 lambda weights (read from CFG, not hardcoded) ──
    fig, ax = plt.subplots(figsize=(11, 5))
    lam = CFG["lambda"]
    bars = ax.bar([f"L_{k}" for k in lam], list(lam.values()),
                  color=[COLORS.get(k, "#888") for k in lam], edgecolor="black", alpha=0.85)
    ax.set_ylim(0, max(lam.values()) * 1.35)
    for b, v in zip(bars, lam.values()):
        _txt_in(ax, b.get_x()+b.get_width()/2, b.get_height()+0.03, f"{v}",
                ha="center", fontweight="bold", fontsize=9)
    ax.set_ylabel("Weight (λ)")
    ax.set_title(f"DCA Loss Weights (from CFG) — λ_unsafe={lam['unsafe']} vs λ_false={lam['false']}")
    plt.setp(ax.get_xticklabels(), rotation=30, ha="right"); _save(fig, "07_lambda_weights.png")

    # ── 08 class distribution (TRAIN split, which is what the weights come from) ──
    if label_dist:
        ls = sorted(label_dist, key=label_dist.get, reverse=True)
        cb = ["#E24B4A" if l in STOP_LABELS else "#534AB7" if l in TURN_STEP else "#1D9E75" for l in ls]
        fig, ax = plt.subplots(figsize=(10, 6))
        ax.barh(ls[::-1], [label_dist[l] for l in ls][::-1], color=cb[::-1], edgecolor="black", alpha=0.85)
        for b, v in zip(ax.patches, [label_dist[l] for l in ls][::-1]):
            ax.text(b.get_width()+3, b.get_y()+b.get_height()/2, str(v), va="center",
                    fontweight="bold", fontsize=10)
        ax.set_xlabel("Frame Count"); ax.set_title("Action Label Distribution — TRAIN split")
        ax.legend(handles=[mpatches.Patch(color="#E24B4A", label="Stop"),
                           mpatches.Patch(color="#534AB7", label="Turn/side"),
                           mpatches.Patch(color="#1D9E75", label="Walk")])
        _save(fig, "08_class_distribution.png")

    # ── 09 mode6 weights ACTUALLY used (v1 plotted 1/sqrt, the code used 1/freq and CB) ──
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 5))
    a1.bar(MODE6_NAMES, freq6, color="#378ADD", edgecolor="black", alpha=0.85)
    a1.set_title("Mode-6 Frequency (train)"); a1.set_ylabel("Count")
    for i, v in enumerate(freq6): a1.text(i, v, f"{int(v)}", ha="center", va="bottom", fontweight="bold", fontsize=9)
    cbv = CB_MODE6_W.tolist()
    a2.bar(MODE6_NAMES, cbv, color="#1D9E75", edgecolor="black", alpha=0.85)
    a2.set_title(f"Class-Balanced Weights ACTUALLY used (β={CFG['cb_beta']})"); a2.set_ylabel("Weight")
    for i, v in enumerate(cbv): a2.text(i, v, f"{v:.2f}", ha="center", va="bottom", fontweight="bold", fontsize=9)
    for a in (a1, a2): plt.setp(a.get_xticklabels(), rotation=25, ha="right")
    fig.suptitle("L_mode Class Balancing (6-class, side-aware)", y=1.03); fig.tight_layout()
    _save(fig, "09_mode_weights.png")

    # ── 10 ladder: raw values, per-panel scale, reference line, unrun rungs SKIPPED ──
    fig, axes = plt.subplots(2, 3, figsize=(17, 10))
    for idx, (mk, ml, better, refk, ceilk) in enumerate(FIG_METRICS):
        ax = axes[idx//3][idx%3]
        vals = [results[r].get(mk, np.nan) for r in rungs]
        bars = ax.bar([RUNG_SHORT.get(r, r) for r in rungs], vals,
                      color=[RUNG_COLORS.get(r, "#888") for r in rungs], edgecolor="black", alpha=0.85)
        anchors = [v for v in vals if np.isfinite(v)]
        has_ref = False
        for k, style, lbl in [(refk, ("--", "#E24B4A"), "trivial"), (ceilk, (":", "#1D9E75"), "ceiling")]:
            rv = ref.get(k) if k else None
            if isinstance(rv, (int, float)) and np.isfinite(rv):
                ax.axhline(rv, ls=style[0], color=style[1], lw=1.6, label=f"{lbl} {rv:.3g}")
                anchors.append(rv); has_ref = True
        if has_ref: ax.legend(fontsize=8, loc="upper left", framealpha=0.85)
        top = max(anchors) * 1.32 if anchors and max(anchors) > 0 else 1.0
        ax.set_ylim(0, top)
        for b, v in zip(bars, vals):
            if np.isfinite(v):
                _txt_in(ax, b.get_x()+b.get_width()/2, v + top*0.02, f"{v:.3g}",
                        ha="center", fontsize=10, fontweight="bold")
        ax.set_title(ml)
    fig.suptitle("Ablation Ladder — raw values with trivial-predictor references", fontsize=16, y=1.01)
    fig.tight_layout(); _save(fig, "10_ablation_ladder.png")

    # ── 11 heatmap of NORMALISED loss (raw was dominated by L_vel) ──
    if epoch_logs.get("dca_full"):
        el = epoch_logs["dca_full"]
        act = [k for k in LOSS_KEYS if any(d.get(k, 0) > 0 for d in el)]
        M = np.array([[d.get(k, 0) for d in el] for k in act], dtype=float)
        M = M / np.maximum(M[:, :1], 1e-9)
        fig, ax = plt.subplots(figsize=(max(8, len(el)), 5))
        im = ax.imshow(M, aspect="auto", cmap="RdYlGn_r", vmin=0, vmax=2)
        ax.set_xticks(range(len(el))); ax.set_xticklabels([f"Ep {d['epoch']}" for d in el])
        ax.set_yticks(range(len(act))); ax.set_yticklabels([f"L_{k}" for k in act])
        for i in range(len(act)):
            for j in range(len(el)):
                ax.text(j, i, f"{M[i,j]:.2f}", ha="center", va="center", fontsize=8)
        plt.colorbar(im, ax=ax, label="L / L(epoch 1)")
        ax.set_title("Loss Heatmap — DCA Full (normalised; <1 = improving, >1 = worsening)")
        _save(fig, "11_loss_heatmap.png")

    # ── 12 architecture (parameterised — v1 hardcoded OpenVLA/SigLIP/Llama-2) ──
    fig, ax = plt.subplots(figsize=(14, 8)); ax.set_xlim(0, 14); ax.set_ylim(0, 10); ax.axis("off")
    def box(x, y, w, h, t, c, a=0.3, fs=11):
        ax.add_patch(plt.Rectangle((x, y), w, h, facecolor=c, alpha=a, edgecolor="black", lw=1.5, zorder=2))
        ax.text(x+w/2, y+h/2, t, ha="center", va="center", fontsize=fs, fontweight="bold", zorder=3)
    def arr(x1, y1, x2, y2, c="black"):
        ax.annotate("", xy=(x2, y2), xytext=(x1, y1), arrowprops=dict(arrowstyle="-|>", color=c, lw=1.5))
    bb = CFG["backbone"]
    box(1, 8.5, 3, 1, "RGB Frame", "#E6F1FB"); box(5, 8.5, 3.5, 1, "Instruction", "#E6F1FB")
    box(0.5, 6, 12, 2, "", "#F1EFE8", a=0.15)
    ax.text(6.5, 7.7, f"{bb} reasoner (frozen + LoRA r={CFG['lora_rank']}, α={CFG['lora_alpha']})",
            ha="center", fontsize=12, fontstyle="italic", color="#5F5E5A")
    box(1, 6.3, 3, 1.2, "Vision Encoder", "#EEEDFE"); box(5.5, 6.3, 3, 1.2, "Projector", "#F1EFE8")
    box(9.5, 6.3, 2.5, 1.2, "LM + LoRA", "#E1F5EE")
    arr(2.5, 8.5, 2.5, 7.5); arr(6.75, 8.5, 6.75, 7.5); arr(4, 6.9, 5.5, 6.9); arr(8.5, 6.9, 9.5, 6.9)
    box(1, 4, 4, 1.2, "z_adapt (LoRA ON)", "#E1F5EE")
    box(8, 4, 4, 1.2, "z₀ (LoRA OFF, cached)", "#F1EFE8")
    arr(5, 6.3, 3, 5.2); arr(10.5, 6.3, 10, 5.2)
    box(3.5, 2.2, 6, 1.2, "Risk Gate: h = z₀ + g·(z_a − z₀)", "#FAEEDA")
    arr(3, 4, 5, 3.4); arr(10, 4, 8, 3.4)
    for x, t, c in [(0.2, "Vel Head\n5-bin ordinal", "#FAECE7"), (3.0, "Dir Head", "#E6F1FB"),
                    (5.8, "Mode Head\n6-class", "#EEEDFE"), (8.6, "Gate Head", "#FBEAF0"),
                    (11.4, "Affordance\n×3", "#E1F5EE")]:
        box(x, 0.3, 2.4, 1.2, t, c, fs=9)
    for xs, xe in [(4.5, 1.4), (5.2, 4.2), (6.5, 7.0), (7.8, 9.8), (8.8, 12.6)]:
        arr(xs, 2.2, xe, 1.5)
    ax.set_title(f"DCA Architecture — {bb}", fontsize=16, pad=15)
    _save(fig, "12_dca_architecture.png")

    # ── 13 roadmap (current backbone marked Done, not "Planned") ──
    fig, ax = plt.subplots(figsize=(12, 6)); ax.set_xlim(0, 12); ax.set_ylim(0, 7); ax.axis("off")
    stages = [("OpenVLA\n(k=0)", "openvla"), ("NaVILA\n(k=7)", "navila"),
              ("VLN-R1", "vln-r1"), ("Cosmos3-Nano", "cosmos3-nano")]
    for i, (t, key) in enumerate(stages):
        x = 1 + i*3
        cur = key in CFG["backbone"]
        box(x, 5, 2.2, 1.2, t, "#E1F5EE" if cur else "#EEEDFE")
        ax.text(x+1.1, 4.7, "RUNNING" if cur else "planned", ha="center", fontsize=10,
                color="#1D9E75" if cur else "#888", fontweight="bold")
    box(3, 2, 6, 1.5, "DCA Block (model-agnostic)\nSwap encode() + command_latent()", "#FAEEDA")
    for i in range(len(stages)): arr(1 + i*3 + 1.1, 5, 6, 3.5)
    ax.set_title("Model Roadmap — Same DCA, Different Hosts", fontsize=14, pad=10)
    _save(fig, "13_model_roadmap.png")

    # ── 14 metric progression with reference bands ──
    fig, axes = plt.subplots(2, 3, figsize=(17, 10))
    for idx, (mk, ml, better, refk, ceilk) in enumerate(FIG_METRICS):
        ax = axes[idx//3][idx%3]
        vals = [results[r].get(mk, np.nan) for r in rungs]
        xp = range(len(rungs))
        ax.plot(xp, vals, "o-", color="#1D9E75", lw=2.5, ms=10, zorder=5)
        anchors = [v for v in vals if np.isfinite(v)]
        rv = ref.get(refk) if refk else None
        if isinstance(rv, (int, float)) and np.isfinite(rv):
            ax.axhline(rv, ls="--", color="#E24B4A", lw=1.6, label=f"trivial {rv:.3g}")
            ax.legend(fontsize=8, loc="upper left", framealpha=0.85)
            anchors.append(rv)
            if better == "close":
                ax.axhspan(rv*0.75, rv*1.25, color="#1D9E75", alpha=0.08)
            else:
                good_lo, good_hi = (0, rv) if better == "min" else (rv, max(anchors)*1.2)
                ax.axhspan(good_lo, good_hi, color="#1D9E75", alpha=0.06)
        top = max(anchors)*1.25 if anchors and max(anchors) > 0 else 1.0
        ax.set_ylim(0, top)
        for xi, v in zip(xp, vals):
            if np.isfinite(v): _txt_in(ax, xi, v + top*0.03, f"{v:.3g}", ha="center",
                                       fontweight="bold", fontsize=10)
        ax.set_xticks(list(xp)); ax.set_xticklabels([RUNG_SHORT.get(r, r) for r in rungs], fontsize=9)
        ax.set_title(ml, fontsize=13)
    fig.suptitle("Metric Progression (shaded = better than the trivial predictor)", fontsize=16, y=1.01)
    fig.tight_layout(); _save(fig, "14_metric_progression.png")

    # ── 15 incremental delta ──
    fig, axes = plt.subplots(2, 3, figsize=(17, 10))
    for idx, (mk, ml, better, refk, ceilk) in enumerate(FIG_METRICS):
        ax = axes[idx//3][idx%3]
        vals = [results[r].get(mk, np.nan) for r in rungs]
        deltas = [0.0] + [vals[i] - vals[i-1] for i in range(1, len(vals))]
        tgt = ref.get(refk) if (better == "close" and refk) else None
        cols = []
        for i, dv in enumerate(deltas):
            if not np.isfinite(dv) or dv == 0: cols.append("#888780"); continue
            if better == "close" and isinstance(tgt, (int, float)) and np.isfinite(tgt):
                # for a "closeness" metric, better means the gap to the demonstrator shrank
                good = abs(vals[i] - tgt) < abs(vals[i-1] - tgt)
            else:
                good = (dv < 0 and better == "min") or (dv > 0 and better == "max")
            cols.append("#1D9E75" if good else "#E24B4A")
        bars = ax.bar([RUNG_SHORT.get(r, r) for r in rungs], deltas, color=cols,
                      edgecolor="black", alpha=0.85)
        span = max([abs(d) for d in deltas if np.isfinite(d)] + [1e-6])
        ax.set_ylim(-span*1.6, span*1.6)
        for b, dv in zip(bars, deltas):
            if np.isfinite(dv) and dv != 0:
                _txt_in(ax, b.get_x()+b.get_width()/2, dv + np.sign(dv)*span*0.12,
                        f"{dv:+.3g}", ha="center", fontsize=9, fontweight="bold")
        ax.axhline(0, color="black", lw=0.6)
        ax.set_title(ml, fontsize=13)
        plt.setp(ax.get_xticklabels(), fontsize=9)
    fig.suptitle("Incremental Delta per Rung (green = better, red = worse)", fontsize=16, y=1.01)
    fig.tight_layout(); _save(fig, "15_incremental_delta.png")

    # ── 16 safety trajectory in (too_slow, too_fast) ──
    # v1 plotted unsafe_go vs stop_recall, which are algebraically y = 1 - x: a straight
    # line by construction that carried no information at all.
    fig, ax = plt.subplots(figsize=(8, 8))
    xs = [results[r].get("too_slow", np.nan) for r in rungs]
    ys = [results[r].get("too_fast", np.nan) for r in rungs]
    ax.plot(xs, ys, "o-", color="#1D9E75", lw=2, ms=12, zorder=5)
    for i, r in enumerate(rungs):
        if np.isfinite(xs[i]) and np.isfinite(ys[i]):
            ax.annotate(RUNG_SHORT.get(r, r), (xs[i]+0.012, ys[i]+0.012), fontsize=11,
                        fontweight="bold", color=RUNG_COLORS.get(r, "#333"))
    ax.scatter([0], [0], s=280, marker="*", color="#BA7517", zorder=6, edgecolors="black")
    ax.annotate("demonstrator\n(0, 0)", (0.02, 0.02), fontsize=11, fontweight="bold", color="#BA7517")
    ax.set_xlabel("Too Slow — paralysis (lower = better)", fontsize=12)
    ax.set_ylabel("Too Fast — reckless (lower = better)", fontsize=12)
    ax.set_title("Safety Trajectory\nboth axes lower = closer to the demonstrator", fontsize=14)
    ax.set_xlim(-0.05, 1.05); ax.set_ylim(-0.05, 1.05)
    ax.axhline(0.5, color="gray", ls="--", alpha=0.3); ax.axvline(0.5, color="gray", ls="--", alpha=0.3)
    _save(fig, "16_safety_trajectory.png")

    # ── 17 results table (with reference row) ──
    cols = ["Rung", "Terms"] + [m[1] for m in FIG_METRICS]
    # NB: lora_unsafe_prior has gate=False. Every notebook in this project labelled it
    # "+gate+prior"; the gate term only ever fires in dca_full. Verified from
    # ABLATION_FLAGS and from which loss keys are populated at train time.
    desc = {"lora_only":"vel+dir+mode", "lora_unsafe":"+unsafe+false",
            "lora_unsafe_prior":"+prior", "dca_full":"+gate+temp (all)"}
    rows = [[RUNG_SHORT.get(r, r), desc.get(r, "")] +
            [f"{results[r].get(m[0], float('nan')):.3g}" for m in FIG_METRICS] for r in rungs]
    refrow = ["TRIVIAL", "baseline"]
    for mk, ml, better, refk, ceilk in FIG_METRICS:
        rv = ref.get(refk) if refk else (ref.get(ceilk) if ceilk else None)
        refrow.append(f"{rv:.3g}" if isinstance(rv, (int, float)) and np.isfinite(rv) else "—")
    rows.append(refrow)
    fig, ax = plt.subplots(figsize=(15, 1.1 + 0.6*len(rows))); ax.axis("off")
    tb = ax.table(cellText=rows, colLabels=cols, loc="center", cellLoc="center")
    tb.auto_set_font_size(False); tb.set_fontsize(10); tb.scale(1, 1.7)
    for j in range(len(cols)):
        tb[0, j].set_facecolor("#E1F5EE"); tb[0, j].set_text_props(fontweight="bold")
        tb[len(rows), j].set_facecolor("#F1EFE8"); tb[len(rows), j].set_text_props(fontstyle="italic")
    for cj, (mk, ml, better, refk, ceilk) in enumerate(FIG_METRICS):
        vs = [results[r].get(mk, np.nan) for r in rungs]
        fin = [v for v in vs if np.isfinite(v)]
        if not fin: continue
        tgt = ref.get(refk) if (better == "close" and refk) else None
        if better == "close" and isinstance(tgt, (int, float)) and np.isfinite(tgt):
            bi = min(range(len(vs)), key=lambda i: abs(vs[i]-tgt) if np.isfinite(vs[i]) else 9e9)
        else:
            bi = vs.index(min(fin) if better == "min" else max(fin))
        tb[bi + 1, cj + 2].set_facecolor("#C8F7DC")
    ax.set_title("Ablation Results Summary (last row = trivial-predictor reference)",
                 fontsize=15, pad=18)
    _save(fig, "17_results_table.png")

    # ── 18 per-class F1 — shows exactly which classes are dead ──
    fig, ax = plt.subplots(figsize=(13, 6))
    x = np.arange(len(LABELS)); w = 0.8 / max(1, len(rungs))
    ax.set_ylim(0, 1.12)
    for i, r in enumerate(rungs):
        pc = results[r].get("per_class_f1", {})
        vals = [pc.get(l, 0.0) for l in LABELS]
        bars = ax.bar(x + i*w, vals, w, label=RUNG_SHORT.get(r, r),
                      color=RUNG_COLORS.get(r, "#888"), alpha=0.88, edgecolor="black")
        for b, v in zip(bars, vals):
            _txt_in(ax, b.get_x()+b.get_width()/2, v+0.02, f"{v:.2f}",
                    ha="center", fontsize=7, fontweight="bold")
    ax.set_xticks(x + w*(len(rungs)-1)/2)
    ax.set_xticklabels(LABELS, rotation=25, ha="right")
    ax.set_ylabel("F1"); ax.set_title("Per-Class F1 — which of the 8 commands the model can actually emit")
    ax.legend(); _save(fig, "18_per_class_f1.png")

    # ── 19 affordance panel (rung-INVARIANT by construction, shown separately) ──
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
    for ax, (mk, ml) in zip(axes, AFFORD_METRICS):
        vals = [results[r].get(mk, np.nan) for r in rungs]
        bars = ax.bar([RUNG_SHORT.get(r, r) for r in rungs], vals,
                      color=[RUNG_COLORS.get(r, "#888") for r in rungs], edgecolor="black", alpha=0.85)
        ax.set_ylim(0, 1.15)
        for b, v in zip(bars, vals):
            if np.isfinite(v): _txt_in(ax, b.get_x()+b.get_width()/2, v+0.02, f"{v:.3f}",
                                       ha="center", fontweight="bold", fontsize=9)
        ax.set_title(ml); plt.setp(ax.get_xticklabels(), fontsize=8, rotation=15, ha="right")
    fig.suptitle("Affordance heads — trained in EVERY rung and read the raw latent, so these are\n"
                 "rung-invariant by construction. They are a sanity check, not an ablation axis.",
                 fontsize=12, y=1.06)
    fig.tight_layout(); _save(fig, "19_affordance_panel.png")

    # ── 20 statistical power ──
    fig, ax = plt.subplots(figsize=(9, 5.5))
    n = max(1, ref.get("n_stop_frames", 1))
    srs = [results[r].get("stop_recall", np.nan) for r in rungs]
    err = [1.96*math.sqrt(max(s*(1-s), 1e-9)/n) if np.isfinite(s) else 0 for s in srs]
    ax.bar([RUNG_SHORT.get(r, r) for r in rungs], srs, yerr=err, capsize=8,
           color=[RUNG_COLORS.get(r, "#888") for r in rungs], edgecolor="black", alpha=0.85)
    ax.set_ylim(0, 1.05); ax.set_ylabel("Stop-Recall")
    ax.set_title(f"Stop-Recall with 95% CI (n = {n} stop frames in {ref.get('n_clips','?')} test clips)\n"
                 "frames inside a clip are correlated, so the true interval is WIDER than shown",
                 fontsize=12)
    plt.setp(ax.get_xticklabels(), fontsize=9); _save(fig, "20_stop_recall_power.png")

    return made

# ============================================================================
# RENDER SMOKE TEST -- would have caught the v1 fig-05 canvas explosion
# ============================================================================
def _smoke_figures():
    import tempfile, random as _r
    _r.seed(0)
    def fake(seed):
        _r.seed(seed)
        return {"macro_f1": _r.uniform(0.1,0.4), "stop_recall": _r.uniform(0,0.6),
                "false_stop": 0.01, "vel_mae": _r.uniform(0.2,0.6),
                "dir_err_deg": _r.uniform(15,25), "jitter": _r.uniform(0.1,0.3),
                "tracks": _r.uniform(0.2,0.6), "too_fast": 0.3, "too_slow": 0.2,
                "turn_side_acc": 0.5, "mode_acc": 0.5,
                "obstacle_auc": 0.97, "proximity_r2": 0.8, "trav_r2": 0.7,
                "per_class_f1": {l: _r.uniform(0,1) for l in LABELS},
                "ref_macro_f1_trivial": 0.084, "ref_macro_f1_mode_oracle": 0.719,
                "ref_vel_mae_median": 0.308, "ref_dir_err_straight": 17.22,
                "ref_jitter_demo": 0.25, "ref_tracks_demo": 1.0,
                "n_stop_frames": 91, "n_clips": 25}
    res  = {r: fake(i) for i, r in enumerate(CFG["rungs"][:2])}   # 2 of 4 -> unrun-rung path
    logs = {"dca_full": [{"epoch": e, "total": 3.0/e, "vel": 1.2/e, "dir": 0.1,
                          "mode": 0.9/e, "consist": 0.05, "unsafe": 0.02, "false": 0.0,
                          "gate": 0.7/e, "prior": 0.08, "temp": 0.02*e} for e in range(1, 6)]}
    with tempfile.TemporaryDirectory() as td:
        made = make_figures(res, logs, td, label_dist={l: 100+i*50 for i, l in enumerate(LABELS)})
        # the v1 bug produced ~20:1; a wide results table is legitimately ~4:1, so 6.0
        # separates "runaway canvas" from "wide figure" without false alarms.
        bad = []
        for p in made:
            w, h = PILImage.open(p).size
            if max(w, h) / min(w, h) > 6.0 or max(w, h) > 8000:
                bad.append(f"{os.path.basename(p)} {w}x{h}")
        assert not bad, "FIGURE SMOKE TEST FAILED — runaway canvas: " + "; ".join(bad)
    print(f"FIGURE SMOKE TEST: PASS ({len(made)} figures, all aspect ratios sane)")

_smoke_figures()

FIG_DIR = os.path.join(CFG["out"], "figures")
_tr_labels = Counter(f["label"] for c in train_clips for f in c["frames"])
paths = make_figures(ALL_RESULTS, ALL_EPOCH_LOGS, FIG_DIR, label_dist=dict(_tr_labels))
print(f"\n{len(paths)} figures saved to {FIG_DIR}/")
for p in sorted(paths): print("  " + os.path.basename(p))

## Part H — Upload to HuggingFace

In [ ]:
# ============================================================================
# UPLOAD SWEEP RESULTS TO HUGGINGFACE
# ============================================================================
import time
from huggingface_hub import HfApi
RUN_ID = time.strftime("run_%Y%m%d_%H%M%S")
api = HfApi()
api.upload_folder(
    folder_path=CFG["out"],
    path_in_repo=f"dca_results/{CFG['backbone']}_v2/{RUN_ID}",
    repo_id=CFG["repo_id"], repo_type="dataset", token=HF_TOKEN,
    commit_message=f"DCA v2 sweep {CFG['backbone']} {RUN_ID} -- phases x rungs")
print(f"uploaded -> {CFG['repo_id']}/dca_results/{CFG['backbone']}_v2/{RUN_ID}")

## Output structure

```
$out/
├── V2_full/
│   ├── lora_only/           adapter_final/ adapter_best/ heads_final.pt heads_best.pt epoch_log.json
│   ├── lora_unsafe/         ...
│   ├── lora_unsafe_prior/   ...
│   ├── dca_full/            ...
│   ├── metrics_lora_only.json      (metrics + ref_* trivial baselines + skill_*)
│   └── metrics_dca_full.json
├── predictions_V2_full_<rung>.json
├── sweep_results.json
├── phase_rung_comparison.json
└── figures/                 20 PNGs
```

## Vast.ai run commands

```bash
source /venv/main/bin/activate
export HF_TOKEN="hf_YOUR_TOKEN"
cd /home/user/denseWalk

python3 -c "
import json
nb = json.load(open('cosmos/Cosmos3_Nano_DCA_stable_v2_vastai.ipynb'))
code = [''.join(c['source']) for c in nb['cells'] if c['cell_type'] == 'code']
open('run_dca_v2.py','w').write((chr(10)*2).join(code))
print('converted')
"

tmux new -s dca
source /venv/main/bin/activate
export HF_TOKEN="hf_YOUR_TOKEN"
cd /home/user/denseWalk
python3 run_dca_v2.py 2>&1 | tee dca_v2_run.log
# Ctrl+B then D to detach ; tmux attach -s dca to come back
```

## Compute expectation

`fast_path=True` cuts `dca_full` from 4 backbone forwards per sample to 1 (cached frozen z₀ +
streamed temporal pairs), plus a one-off z₀ build pass over the dataset. On ~7.5k train frames
(215 clips) expect roughly **2–3 h per rung** instead of ~7.5 h, so the full 4-rung ladder lands
around **8–12 h** rather than ~30 h.

In [ ]:
print('='*80)
print('V3 DECISION REPORT')
print('='*80)
import json
for cfg in ['baseline', 'decouple_only', 'all_fixes']:
    for rung in ['lora_only', 'dca_full']:
        try:
            with open(f"{CFG['out']}/{cfg}_{rung}/metrics_final.json") as f:
                m = json.load(f)
            print(f"{cfg:15s} {rung:12s}: f1={m.get('macro_f1',0):.3f} "
                  f"obs_std={m.get('obs_dist_std',0):.3f} prox_r2={m.get('proximity_r2',0):.3f}")
        except:
            print(f"{cfg:15s} {rung:12s}: NOT FOUND")
print('='*80)
